# 3\.4\.1 Evaluate Selected Surface Against NYC Crash Events

In [1]:
%pip install plotly==6.8.0


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ---------------------------------------------------------------------
# Import libraries for selected-surface crash validation
# ---------------------------------------------------------------------

from pathlib import Path
import json

import duckdb
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely import wkb

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from project_branding import (
    BRAND_COLORS,
    BRAND_COLOR_SEQUENCE,
    BRAND_DIVERGING_SEQUENCE,
    BRAND_MAP_COLORS,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = BRAND_COLOR_SEQUENCE

In [3]:
# ---------------------------------------------------------------------
# Define project paths, canonical columns, helper functions, and toggles
# ---------------------------------------------------------------------

PROJECT_ROOT = Path("/datasets/_deepnote_work")
PIPELINE_DATA_DIR = PROJECT_ROOT / "pipeline_data"
SOURCE_DATA_DIR = PROJECT_ROOT / "source_data"

FINAL_336_DIR = PIPELINE_DATA_DIR / "3.3.6.final_tables"
INTERMEDIATE_341_DIR = PIPELINE_DATA_DIR / "3.4.1.intermediate_tables"
FINAL_341_DIR = PIPELINE_DATA_DIR / "3.4.1.final_tables"

INTERMEDIATE_341_DIR.mkdir(parents=True, exist_ok=True)
FINAL_341_DIR.mkdir(parents=True, exist_ok=True)

STUDY_START_DATE = "2023-01-01"
STUDY_END_DATE = "2026-03-31"
CONGESTION_PRICING_START_DATE = "2025-01-05"

EVENT_ID_COLUMN = "comparison_event_id"
ZONE_ID_COLUMN = "taxi_zone_id"
ZONE_COLUMN = "zone"
BOROUGH_COLUMN = "borough"
DATE_COLUMN = "date"
DAYPART_COLUMN = "daypart"
PRE_POST_CP_COLUMN = "pre_post_cp"
TEMPORAL_BUCKET_COLUMN = "temporal_bucket"
POSITIVE_DIRECTION_PROFILE_COLUMN = "positive_direction_profile"
TEMPORAL_BUCKET_ORDER = [
    "weekday_overnight",
    "weekday_am_peak",
    "weekday_midday",
    "weekday_pm_peak",
    "weekday_evening",
    "weekend_overnight",
    "weekend_am_peak",
    "weekend_midday",
    "weekend_pm_peak",
    "weekend_evening",
]
DAYPART_ORDER = ["overnight", "am_peak", "midday", "pm_peak", "evening"]

SELECTED_FINALIST_SURFACE_LABEL = "All 3 retained frameworks"
COMPARISON_FINALIST_SURFACE_LABEL = "2+ retained frameworks"

SELECTED_FINALIST_SURFACE_PATH = FINAL_336_DIR / "selected_finalist_stress_surface.parquet"
SELECTED_FINALIST_SURFACE_SUMMARY_PATH = (
    FINAL_336_DIR / "selected_finalist_stress_surface_summary.csv"
)
SELECTED_FINALIST_EVENT_UNIVERSE_PATH = (
    FINAL_336_DIR / "selected_finalist_event_universe.parquet"
)
SELECTED_FINALIST_FULL_ROW_UNIVERSE_ENRICHED_PATH = (
    FINAL_336_DIR / "selected_finalist_full_row_universe_enriched.parquet"
)
SELECTED_FINALIST_CONTEXTUAL_REVIEW_BACKBONE_PATH = (
    FINAL_336_DIR / "selected_finalist_contextual_review_backbone.parquet"
)
COMPARISON_FINALIST_SURFACE_2PLUS_PATH = (
    FINAL_336_DIR / "comparison_finalist_surface_2plus.parquet"
)
FINALIST_DECISION_SUMMARY_PATH = FINAL_336_DIR / "finalist_decision_summary.csv"
FINALIST_DECISION_EVIDENCE_PATH = FINAL_336_DIR / "finalist_decision_evidence.csv"
FINALIST_HANDOFF_MANIFEST_PATH = FINAL_336_DIR / "finalist_handoff_manifest.csv"
CRASH_EVENTS_CSV_PATH = SOURCE_DATA_DIR / "Motor_Vehicle_Collisions_-_Crashes_20260615.csv"
TAXI_ZONE_SPATIAL_CONTEXT_PATH = (
    PIPELINE_DATA_DIR / "1.3.1.final_tables" / "nyc_taxi_zones_harmonized.parquet"
)
HARMONIZED_TAXI_ZONES_PATH = TAXI_ZONE_SPATIAL_CONTEXT_PATH
TAXI_ZONE_GEOMETRY_PATH = TAXI_ZONE_SPATIAL_CONTEXT_PATH
CRASH_RESCUE_GEOMETRY_CANDIDATE_PATHS = [
    TAXI_ZONE_SPATIAL_CONTEXT_PATH,
    HARMONIZED_TAXI_ZONES_PATH,
    TAXI_ZONE_GEOMETRY_PATH,
]
CRASH_ZONE_BUFFER_CALIBRATION_PATH = (
    INTERMEDIATE_341_DIR / "crash_zone_buffer_calibration.parquet"
)
CRASH_MANUAL_RESCUE_LOOKUP_PATH = (
    INTERMEDIATE_341_DIR / "crash_manual_rescue_lookup.parquet"
)
CRASH_TO_ZONE_LINKAGE_PATH = INTERMEDIATE_341_DIR / "crash_to_zone_linkage.parquet"
CRASH_TO_ZONE_LINKAGE_SUMMARY_PATH = (
    INTERMEDIATE_341_DIR / "crash_to_zone_linkage_summary.parquet"
)
CRASH_CONTEXT_SUMMARY_PATH = INTERMEDIATE_341_DIR / "crash_context_summary.parquet"

REBUILD_341_CRASH_LINKAGE = False
REBUILD_341_VALIDATION = False
WRITE_341_OUTPUTS = True

REBUILD_CRASH_ZONE_BUFFER_CALIBRATION = REBUILD_341_CRASH_LINKAGE
CRASH_RESCUE_GEOMETRY_CANDIDATE_PATH = CRASH_RESCUE_GEOMETRY_CANDIDATE_PATHS


def should_rebuild(flag, output_path):
    return bool(flag) or not Path(output_path).exists()


def assert_unique_key(df, key_columns, object_name):
    duplicate_count = int(df.duplicated(subset=key_columns).sum())
    assert duplicate_count == 0, (
        f"{object_name} must be unique on {key_columns}; found {duplicate_count} duplicates."
    )


def load_parquet_without_duplicate_columns(path_like, columns=None):
    df = pd.read_parquet(path_like, columns=columns)
    return df.loc[:, ~df.columns.duplicated()].copy()


def normalize_bool_flag(series_like):
    series = pd.Series(series_like).copy()
    if series.dtype == bool:
        return series.fillna(False)
    normalized = (
        series.fillna(False)
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )
    return normalized.fillna(False)


def status_label(condition):
    return "pass" if bool(condition) else "review"


def assign_daypart_from_hour(hour):
    if pd.isna(hour):
        return pd.NA

    hour = int(hour)
    if 6 <= hour < 10:
        return "am_peak"
    if 10 <= hour < 16:
        return "midday"
    if 16 <= hour < 20:
        return "pm_peak"
    if 20 <= hour < 24:
        return "evening"
    return "overnight"


def normalize_taxi_zone_geometry_value(value):
    if value is None or pd.isna(value):
        return None
    if hasattr(value, "geom_type"):
        return value
    if isinstance(value, memoryview):
        value = value.tobytes()
    if isinstance(value, (bytes, bytearray)):
        return wkb.loads(bytes(value))
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return None
        try:
            return wkb.loads(bytes.fromhex(value))
        except ValueError:
            return value
    return value


def build_taxi_zone_geometry_gdf(zone_df, source_crs="EPSG:4326"):
    prepared_df = zone_df.loc[:, ~zone_df.columns.duplicated()].copy()

    rename_map = {}
    if ZONE_ID_COLUMN not in prepared_df.columns:
        for candidate in ["LocationID", "location_id", "locationid", "OBJECTID", "objectid"]:
            if candidate in prepared_df.columns:
                rename_map[candidate] = ZONE_ID_COLUMN
                break
    if ZONE_COLUMN not in prepared_df.columns:
        for candidate in ["Zone", "zone_name", "taxi_zone"]:
            if candidate in prepared_df.columns:
                rename_map[candidate] = ZONE_COLUMN
                break
    if BOROUGH_COLUMN not in prepared_df.columns:
        for candidate in ["Borough", "borough_name"]:
            if candidate in prepared_df.columns:
                rename_map[candidate] = BOROUGH_COLUMN
                break

    if rename_map:
        prepared_df = prepared_df.rename(columns=rename_map)

    assert ZONE_ID_COLUMN in prepared_df.columns, (
        f"Taxi-zone table is missing {ZONE_ID_COLUMN}."
    )

    geometry_column = None
    for candidate in ["geometry", "geom", "the_geom", "wkb_geometry"]:
        if candidate in prepared_df.columns:
            geometry_column = candidate
            break

    assert geometry_column is not None, "Taxi-zone table is missing a geometry column."

    prepared_df[ZONE_ID_COLUMN] = pd.to_numeric(
        prepared_df[ZONE_ID_COLUMN],
        errors="coerce",
    )
    prepared_df = prepared_df.loc[prepared_df[ZONE_ID_COLUMN].notna()].copy()
    prepared_df[ZONE_ID_COLUMN] = prepared_df[ZONE_ID_COLUMN].astype(int)

    if ZONE_COLUMN not in prepared_df.columns:
        prepared_df[ZONE_COLUMN] = ""
    if BOROUGH_COLUMN not in prepared_df.columns:
        prepared_df[BOROUGH_COLUMN] = ""

    prepared_df["geometry"] = prepared_df[geometry_column].map(
        normalize_taxi_zone_geometry_value
    )
    prepared_df = prepared_df.loc[prepared_df["geometry"].notna()].copy()

    return gpd.GeoDataFrame(
        prepared_df[[ZONE_ID_COLUMN, ZONE_COLUMN, BOROUGH_COLUMN, "geometry"]]
        .drop_duplicates(subset=[ZONE_ID_COLUMN])
        .copy(),
        geometry="geometry",
        crs=source_crs,
    )


def build_taxi_zone_lookup_df(zone_df):
    zone_gdf = build_taxi_zone_geometry_gdf(zone_df)
    return (
        zone_gdf[[ZONE_ID_COLUMN, ZONE_COLUMN, BOROUGH_COLUMN]]
        .drop_duplicates(subset=[ZONE_ID_COLUMN])
        .copy()
    )

## 3\.4\.1\.1 Load the Selected Finalist Crash\-Validation Backbone

This section loads the finalist handoff package from \`3\.3\.6\` before we touch the crash data\. The goal is to make the crash\-validation target explicit: \`All 3 retained frameworks\` is already the selected downstream surface, and this notebook is here to test whether that selected surface looks externally plausible when compared against mapped NYC crash activity\. We will keep the broader \`2\+ retained frameworks\` surface available only as a robustness comparison, not as a second unresolved finalist\.

Load the \`3\.3\.6\` exports that matter for crash validation\. We need the selected finalist surface itself, the complete event\-universe spine with finalist membership flags, the enriched full\-row universe with stress\-direction and contextual metadata, and the compact decision records that explain what was already chosen upstream\. Once those assets are loaded and validated, the rest of the notebook can focus on crash linkage rather than recreating the finalist selection logic\.

In [4]:
# ---------------------------------------------------------------------
# Load the 3.3.6 finalist handoff package for crash validation
# ---------------------------------------------------------------------

selected_finalist_surface_df = load_parquet_without_duplicate_columns(
    SELECTED_FINALIST_SURFACE_PATH
)
selected_finalist_surface_df = selected_finalist_surface_df[
    [EVENT_ID_COLUMN, ZONE_ID_COLUMN, DATE_COLUMN, DAYPART_COLUMN]
].copy()

selected_finalist_event_universe_df = load_parquet_without_duplicate_columns(
    SELECTED_FINALIST_EVENT_UNIVERSE_PATH
)
selected_finalist_event_universe_df = selected_finalist_event_universe_df[
    [
        EVENT_ID_COLUMN,
        ZONE_ID_COLUMN,
        DATE_COLUMN,
        DAYPART_COLUMN,
        "selected_finalist_flag",
        "comparison_finalist_flag",
        "added_by_2plus_expansion_flag",
    ]
].copy()

selected_finalist_full_row_universe_enriched_df = load_parquet_without_duplicate_columns(
    SELECTED_FINALIST_FULL_ROW_UNIVERSE_ENRICHED_PATH
)
selected_finalist_full_row_universe_enriched_df = (
    selected_finalist_full_row_universe_enriched_df.copy()
)

selected_finalist_contextual_review_backbone_df = load_parquet_without_duplicate_columns(
    SELECTED_FINALIST_CONTEXTUAL_REVIEW_BACKBONE_PATH
)
comparison_finalist_surface_2plus_df = load_parquet_without_duplicate_columns(
    COMPARISON_FINALIST_SURFACE_2PLUS_PATH
)
comparison_finalist_surface_2plus_df = comparison_finalist_surface_2plus_df[
    [EVENT_ID_COLUMN, ZONE_ID_COLUMN, DATE_COLUMN, DAYPART_COLUMN]
].copy()

finalist_decision_summary_df = pd.read_csv(FINALIST_DECISION_SUMMARY_PATH).copy()
finalist_decision_evidence_df = pd.read_csv(FINALIST_DECISION_EVIDENCE_PATH).copy()
finalist_handoff_manifest_df = pd.read_csv(FINALIST_HANDOFF_MANIFEST_PATH).copy()
retained_map_geometry_gdf = load_parquet_without_duplicate_columns(
    TAXI_ZONE_SPATIAL_CONTEXT_PATH
)

assert_unique_key(
    selected_finalist_surface_df,
    [EVENT_ID_COLUMN],
    "selected_finalist_surface_df",
)
assert_unique_key(
    selected_finalist_event_universe_df,
    [EVENT_ID_COLUMN],
    "selected_finalist_event_universe_df",
)
assert_unique_key(
    comparison_finalist_surface_2plus_df,
    [EVENT_ID_COLUMN],
    "comparison_finalist_surface_2plus_df",
)

selected_event_ids = set(selected_finalist_surface_df[EVENT_ID_COLUMN])
comparison_event_ids = set(comparison_finalist_surface_2plus_df[EVENT_ID_COLUMN])

finalist_backbone_inventory_df = pd.DataFrame(
    [
        {
            "object_name": "selected_finalist_surface_df",
            "role": "selected all-3 surface rows",
            "row_count": int(len(selected_finalist_surface_df)),
            "status": status_label(len(selected_finalist_surface_df) > 0),
        },
        {
            "object_name": "comparison_finalist_surface_2plus_df",
            "role": "broader 2+ robustness comparison",
            "row_count": int(len(comparison_finalist_surface_2plus_df)),
            "status": status_label(len(comparison_finalist_surface_2plus_df) > 0),
        },
        {
            "object_name": "selected_finalist_event_universe_df",
            "role": "full event spine with finalist membership flags",
            "row_count": int(len(selected_finalist_event_universe_df)),
            "status": status_label(len(selected_finalist_event_universe_df) > 0),
        },
        {
            "object_name": "selected_finalist_full_row_universe_enriched_df",
            "role": "app-facing full row universe with stress and context metadata",
            "row_count": int(len(selected_finalist_full_row_universe_enriched_df)),
            "status": status_label(len(selected_finalist_full_row_universe_enriched_df) > 0),
        },
    ]
)

finalist_backbone_summary_df = pd.DataFrame(
    [
        {
            "selected_surface_label": SELECTED_FINALIST_SURFACE_LABEL,
            "selected_surface_rows": int(len(selected_finalist_surface_df)),
            "comparison_surface_label": COMPARISON_FINALIST_SURFACE_LABEL,
            "comparison_surface_rows": int(len(comparison_finalist_surface_2plus_df)),
            "incremental_rows_vs_selected": int(
                len(comparison_event_ids.difference(selected_event_ids))
            ),
            "event_universe_rows": int(len(selected_finalist_event_universe_df)),
            "selected_flag_rows_in_universe": int(
                selected_finalist_event_universe_df["selected_finalist_flag"].sum()
            ),
            "comparison_flag_rows_in_universe": int(
                selected_finalist_event_universe_df["comparison_finalist_flag"].sum()
            ),
        }
    ]
)

display(finalist_backbone_inventory_df)
display(finalist_backbone_summary_df)
display(finalist_decision_summary_df)

,object_name,role,row_count,status
0,selected_finalist_surface_df,selected all-3 surface rows,155353,pass
1,comparison_finalist_surface_2plus_df,broader 2+ robustness comparison,299439,pass
2,selected_finalist_event_universe_df,full event spine with finalist membership flags,1559590,pass
3,selected_finalist_full_row_universe_enriched_df,app-facing full row universe with stress and context metadata,1559590,pass


,selected_surface_label,selected_surface_rows,comparison_surface_label,comparison_surface_rows,incremental_rows_vs_selected,event_universe_rows,selected_flag_rows_in_universe,comparison_flag_rows_in_universe
0,All 3 retained frameworks,155353,2+ retained frameworks,299439,144086,1559590,155353,299439


,selected_surface_id,selected_surface_label,comparison_surface_id,comparison_surface_label,transition_surface_label,selected_event_count,comparison_event_count,incremental_event_count,decision_reason
0,framework_consensus_3way,All 3 retained frameworks,framework_consensus_2plus,2+ retained frameworks,Added by 2+ expansion,155353,299439,144086,"Selected the strict all-3 surface because it preserves the main citywide stress story while remaining more selective, more coherent, and less expansionary than the broader 2+ finalist."


Findings\. The crash\-validation backbone is now anchored to the \`3\.3\.6\` handoff rather than the legacy retained\-positive\-direction package\. \`All 3 retained frameworks\` enters this notebook as the already\-selected downstream surface, the full event spine is available with explicit finalist membership flags, and the enriched full\-row universe carries the contextual and stress\-direction metadata we will need once crash linkage is attached\.

## 3\.4\.1\.2 Build and Evaluate the Crash\-Linked Validation Layer

This section brings in an outside reality check by comparing the selected finalist surface against NYC crash\-event data\. Up to this point, we have shown that the selected \`All 3 retained frameworks\` package is internally coherent, directionally interpretable, and plausibly relevant to congestion\-pricing analysis\. The next question is whether that selected surface lines up at all with an independently collected signal of real\-world transportation disruption\.

The goal here is not to build a crash\-prediction model or to claim that every anomaly should correspond to a crash\. Crash records are noisy, incomplete, and only capture one slice of street\-level disruption\. The point is narrower: if the selected \`All 3 retained frameworks\` surface, and the broader \`2\+ retained frameworks\` robustness comparator, show a plausible relationship to mapped crash contexts, that gives the upstream finalist decision added external credibility without reopening it\.

### Read and profile the crash data

We begin by reading and profiling the crash data itself before merging it into the anomaly evaluation layer\. That step matters because the usefulness of the external validation depends on how complete, linkable, and temporally aligned the crash records actually are\. This block loads the raw NYC crash\-event data and performs a first\-pass inventory of its structure\. The goal is simply to confirm that the file is present, readable, and large enough to support external validation before we start making linkage assumptions\.

In [5]:
# ---------------------------------------------------------------------
# Load the raw NYC crash-event data
# ---------------------------------------------------------------------

crash_raw_df = pd.read_csv(CRASH_EVENTS_CSV_PATH, low_memory=False)

crash_file_inventory_df = pd.DataFrame(
    [
        {
            "crash_file_path": str(CRASH_EVENTS_CSV_PATH),
            "rows": len(crash_raw_df),
            "columns": len(crash_raw_df.columns),
            "status": "pass" if len(crash_raw_df) > 0 else "review",
        }
    ]
)

display(crash_file_inventory_df)
display(pd.DataFrame({"crash_columns": crash_raw_df.columns.tolist()}))

assert crash_file_inventory_df["status"].eq("pass").all(), (
    "Crash file loaded but appears empty."
)

,crash_file_path,rows,columns,status
0,/datasets/_deepnote_work/source_data/Motor_Vehicle_Collisions_-_Crashes_20260615.csv,309769,29,pass


,crash_columns
0,CRASH DATE
1,CRASH TIME
2,BOROUGH
3,ZIP CODE
4,LATITUDE
5,LONGITUDE
6,LOCATION
7,ON STREET NAME
8,CROSS STREET NAME
9,OFF STREET NAME


Findings\. The crash file is large enough to support a meaningful external check, with 309,769 raw rows and 29 columns\. At a glance, it already appears to contain the key ingredients we would hope for: event date and time, coordinates, borough context, injury and fatality counts, and a unique collision ID\. So the raw crash data looks promising as an external\-validation source; the next question is how much of it is complete and clean enough to align to our anomaly grain\.

This block profiles the raw crash fields so we can see what location, time, and severity information is available before we try to standardize anything\. The point is to understand the raw ingredients, not to force a linkage too early\.

In [6]:
# ---------------------------------------------------------------------
# Profile raw crash-data field availability
# ---------------------------------------------------------------------

raw_crash_field_profile_df = pd.DataFrame(
    {
        "column_name": crash_raw_df.columns,
        "non_null_rows": [crash_raw_df[col].notna().sum() for col in crash_raw_df.columns],
        "null_rows": [crash_raw_df[col].isna().sum() for col in crash_raw_df.columns],
        "null_share": [crash_raw_df[col].isna().mean() for col in crash_raw_df.columns],
    }
).sort_values(["null_share", "column_name"]).reset_index(drop=True)

display(raw_crash_field_profile_df.style.format({"null_share": "{:.3f}"}))

,column_name,non_null_rows,null_rows,null_share
0,COLLISION_ID,309769,0,0.000
1,CRASH DATE,309769,0,0.000
2,CRASH TIME,309769,0,0.000
3,NUMBER OF CYCLIST INJURED,309769,0,0.000
4,NUMBER OF CYCLIST KILLED,309769,0,0.000
5,NUMBER OF MOTORIST INJURED,309769,0,0.000
6,NUMBER OF MOTORIST KILLED,309769,0,0.000
7,NUMBER OF PEDESTRIANS INJURED,309769,0,0.000
8,NUMBER OF PEDESTRIANS KILLED,309769,0,0.000
9,NUMBER OF PERSONS INJURED,309769,0,0.000


Findings\. The raw crash file looks strong where we need it to be strongest\. The core linkage fields are fully present: COLLISION\_ID, CRASH DATE, and CRASH TIME have no missing values, and the main injury\-count fields are also complete\. The coordinate fields are in good shape too, with LATITUDE, LONGITUDE, and LOCATION populated for about 95% of rows\. That means the crash data is in much better condition for external validation than for fine\-grained street\-description analysis\.

Standardize Timestamps\. This block standardizes crash date and time into a clean event timestamp and derives the notebook’s temporal buckets\. That lets us test whether the crash data can be aligned to the same daypart structure used by the anomaly surfaces\.

In [7]:
# ---------------------------------------------------------------------
# Standardize crash dates/times and derive dayparts
# ---------------------------------------------------------------------

required_columns = [
    "COLLISION_ID",
    "CRASH DATE",
    "CRASH TIME",
]
missing_columns = [c for c in required_columns if c not in crash_raw_df.columns]
assert not missing_columns, (
    f"crash_raw_df is missing required columns: {missing_columns}"
)

crash_profile_df = crash_raw_df.copy()

crash_profile_df["crash_date"] = pd.to_datetime(
    crash_profile_df["CRASH DATE"],
    errors="coerce",
)
crash_profile_df[DATE_COLUMN] = crash_profile_df["crash_date"].dt.normalize()

crash_time_string = crash_profile_df["CRASH TIME"].astype(str).str.strip()
crash_profile_df["crash_time_parsed"] = pd.to_datetime(
    crash_time_string,
    format="%H:%M",
    errors="coerce",
)

crash_profile_df["crash_hour"] = crash_profile_df["crash_time_parsed"].dt.hour
crash_profile_df["crash_minute"] = crash_profile_df["crash_time_parsed"].dt.minute
crash_profile_df["crash_dayofweek"] = crash_profile_df["crash_date"].dt.dayofweek
crash_profile_df["is_weekend"] = crash_profile_df["crash_dayofweek"].ge(5)

crash_profile_df[DAYPART_COLUMN] = [
    assign_daypart_from_hour(hour) for hour in crash_profile_df["crash_hour"]
]
crash_profile_df[DAYPART_COLUMN] = pd.Categorical(
    crash_profile_df[DAYPART_COLUMN],
    categories=DAYPART_ORDER,
    ordered=True,
)

# Preserve the richer temporal bucket as optional context, but keep
# plain daypart as the core crash-validation grain used downstream.
def assign_temporal_bucket(is_weekend, hour):
    if pd.isna(hour):
        return pd.NA

    prefix = "weekend" if bool(is_weekend) else "weekday"
    suffix = assign_daypart_from_hour(hour)
    return f"{prefix}_{suffix}"


crash_profile_df[TEMPORAL_BUCKET_COLUMN] = [
    assign_temporal_bucket(is_weekend, hour)
    for is_weekend, hour in zip(
        crash_profile_df["is_weekend"],
        crash_profile_df["crash_hour"],
    )
]
crash_profile_df[TEMPORAL_BUCKET_COLUMN] = pd.Categorical(
    crash_profile_df[TEMPORAL_BUCKET_COLUMN],
    categories=TEMPORAL_BUCKET_ORDER,
    ordered=True,
)

crash_daypart_inventory_df = (
    crash_profile_df.groupby(DAYPART_COLUMN, dropna=False, observed=False)
    .agg(crash_rows=("COLLISION_ID", "size"))
    .reset_index()
    .sort_values("crash_rows", ascending=False)
    .reset_index(drop=True)
)

display(crash_daypart_inventory_df)

assert crash_profile_df[DAYPART_COLUMN].notna().any(), (
    "No crash rows were assigned to a daypart. Please inspect CRASH DATE / CRASH TIME parsing."
)

,daypart,crash_rows
0,midday,94115
1,pm_peak,71156
2,am_peak,51172
3,evening,47997
4,overnight,45329


Findings\. After standardizing the crash timestamps to plain dayparts, the crash data still spans the full daily cycle used by the selected finalist universe\. midday is the largest bucket with 94,115 crash rows, followed by pm\_peak at 71,156, am\_peak at 51,172, evening at 47,997, and overnight at 45,329\. So the crash layer is now aligned to the notebook’s row grain without losing meaningful temporal coverage\.

This block checks how complete the spatial side of the crash data is\. We want to know whether enough crashes have valid coordinates to support a taxi\-zone linkage, since that will determine whether spatial buffering is worth doing\.

In [8]:
# ---------------------------------------------------------------------
# Profile crash-coordinate completeness
# ---------------------------------------------------------------------

required_columns = ["LATITUDE", "LONGITUDE", "COLLISION_ID"]
missing_columns = [c for c in required_columns if c not in crash_profile_df.columns]
assert not missing_columns, (
    f"crash_profile_df is missing required columns: {missing_columns}"
)

crash_profile_df["latitude_numeric"] = pd.to_numeric(
    crash_profile_df["LATITUDE"],
    errors="coerce",
)
crash_profile_df["longitude_numeric"] = pd.to_numeric(
    crash_profile_df["LONGITUDE"],
    errors="coerce",
)

crash_profile_df["has_valid_coordinates"] = (
    crash_profile_df["latitude_numeric"].between(40.0, 41.5, inclusive="both")
    & crash_profile_df["longitude_numeric"].between(-75.5, -73.0, inclusive="both")
)
crash_profile_df["has_valid_coordinates"] = crash_profile_df["has_valid_coordinates"].fillna(False)

crash_coordinate_profile_df = pd.DataFrame(
    [
        {
            "raw_rows": int(len(crash_profile_df)),
            "rows_with_valid_coordinates": int(crash_profile_df["has_valid_coordinates"].sum()),
            "coordinate_share": float(crash_profile_df["has_valid_coordinates"].mean()),
        }
    ]
)

display(crash_coordinate_profile_df)

assert crash_coordinate_profile_df["rows_with_valid_coordinates"].iloc[0] > 0, (
    "No valid crash coordinates were detected. Please inspect LATITUDE / LONGITUDE parsing."
)

,raw_rows,rows_with_valid_coordinates,coordinate_share
0,309769,290568,0.938015


Findings\. The crash file is spatially usable\. About 93\.8% of raw crash rows have valid coordinates, which is strong enough to support a taxi\-zone linkage step later on\. At this stage we are only checking coordinate availability, not yet assigning crashes to zones\. The actual linkage will still need a small spatial buffer so crashes near bridges, tunnels, highways, or water edges can register against nearby taxi zones instead of being dropped just because the point falls slightly outside a zone polygon\.

Missing Coordinates\. This block takes a closer look at the crash rows that are missing coordinates\. The goal is to check whether those missing\-location crashes look random or whether they cluster in specific roadway contexts, such as bridges, highways, or off\-street records\. That matters because if the missingness is systematic, then the crash\-validation layer may underrepresent certain types of transportation disruption even before any taxi\-zone linkage begins\.

In [9]:
# ---------------------------------------------------------------------
# Profile what kinds of crash records are missing coordinates
# ---------------------------------------------------------------------

required_columns = [
    "has_valid_coordinates",
    "BOROUGH",
    "ON STREET NAME",
    "CROSS STREET NAME",
    "OFF STREET NAME",
]
missing_columns = [c for c in required_columns if c not in crash_profile_df.columns]
assert not missing_columns, (
    f"crash_profile_df is missing required columns: {missing_columns}"
)

missing_coord_df = crash_profile_df.loc[
    ~crash_profile_df["has_valid_coordinates"].fillna(False)
].copy()

for street_col in ["ON STREET NAME", "CROSS STREET NAME", "OFF STREET NAME"]:
    missing_coord_df[street_col] = missing_coord_df[street_col].fillna("").astype(str).str.strip()

combined_text = (
    missing_coord_df["ON STREET NAME"] + " "
    + missing_coord_df["CROSS STREET NAME"] + " "
    + missing_coord_df["OFF STREET NAME"]
).str.upper()

missing_coord_df["bridge_named"] = combined_text.str.contains("BRIDGE", na=False)
missing_coord_df["tunnel_named"] = combined_text.str.contains("TUNNEL", na=False)
missing_coord_df["highway_or_expressway_named"] = combined_text.str.contains(
    "HIGHWAY|EXPRESSWAY|PARKWAY|THRUWAY|FDR DRIVE|HARLEM RIVER DRIVE",
    regex=True,
    na=False,
)
missing_coord_df["ramp_or_entry_exit_named"] = combined_text.str.contains(
    "RAMP|EXIT|ENTRANCE|ENTRY",
    regex=True,
    na=False,
)
missing_coord_df["special_roadway_context"] = (
    missing_coord_df[
        [
            "bridge_named",
            "tunnel_named",
            "highway_or_expressway_named",
            "ramp_or_entry_exit_named",
        ]
    ]
    .any(axis=1)
)

missing_coordinate_summary_df = pd.DataFrame(
    [
        {
            "raw_rows": int(len(crash_profile_df)),
            "missing_coordinate_rows": int(len(missing_coord_df)),
            "missing_coordinate_share": len(missing_coord_df) / len(crash_profile_df) if len(crash_profile_df) else np.nan,
            "special_roadway_missing_rows": int(missing_coord_df["special_roadway_context"].sum()),
            "special_roadway_share_within_missing": (
                missing_coord_df["special_roadway_context"].mean()
                if len(missing_coord_df) else np.nan
            ),
        }
    ]
)

missing_coordinate_context_df = pd.DataFrame(
    [
        {
            "context_flag": "bridge_named",
            "crash_rows": int(missing_coord_df["bridge_named"].sum()),
            "share_within_missing_coord_rows": missing_coord_df["bridge_named"].mean() if len(missing_coord_df) else np.nan,
        },
        {
            "context_flag": "tunnel_named",
            "crash_rows": int(missing_coord_df["tunnel_named"].sum()),
            "share_within_missing_coord_rows": missing_coord_df["tunnel_named"].mean() if len(missing_coord_df) else np.nan,
        },
        {
            "context_flag": "highway_or_expressway_named",
            "crash_rows": int(missing_coord_df["highway_or_expressway_named"].sum()),
            "share_within_missing_coord_rows": missing_coord_df["highway_or_expressway_named"].mean() if len(missing_coord_df) else np.nan,
        },
        {
            "context_flag": "ramp_or_entry_exit_named",
            "crash_rows": int(missing_coord_df["ramp_or_entry_exit_named"].sum()),
            "share_within_missing_coord_rows": missing_coord_df["ramp_or_entry_exit_named"].mean() if len(missing_coord_df) else np.nan,
        },
    ]
)

location_string_records = []
for source_field in ["ON STREET NAME", "CROSS STREET NAME", "OFF STREET NAME"]:
    top_counts = (
        missing_coord_df[source_field]
        .replace("", pd.NA)
        .fillna("(missing)")
        .value_counts(dropna=False)
        .head(10)
        .reset_index()
    )
    top_counts.columns = ["location_string", "crash_rows"]
    top_counts["source_field"] = source_field
    location_string_records.append(top_counts)

missing_coordinate_location_examples_df = pd.concat(
    location_string_records,
    ignore_index=True,
)

display(missing_coordinate_summary_df.round(3))
display(missing_coordinate_context_df.round(3))
display(missing_coordinate_location_examples_df)

,raw_rows,missing_coordinate_rows,missing_coordinate_share,special_roadway_missing_rows,special_roadway_share_within_missing
0,309769,19201,0.062,7623,0.397


,context_flag,crash_rows,share_within_missing_coord_rows
0,bridge_named,2633,0.137
1,tunnel_named,190,0.010
2,highway_or_expressway_named,4640,0.242
3,ramp_or_entry_exit_named,894,0.047


,location_string,crash_rows,source_field
0,(missing),4633,ON STREET NAME
1,VERRAZANO BRIDGE UPPER,549,ON STREET NAME
2,TRIBOROUGH BRIDGE,505,ON STREET NAME
3,BRONX WHITESTONE BRIDGE,293,ON STREET NAME
4,GRAND CENTRAL PARKWAY,258,ON STREET NAME
5,BELT PARKWAY,218,ON STREET NAME
6,THROGS NECK BRIDGE,191,ON STREET NAME
7,VANWYCK EXPRESSWAY,182,ON STREET NAME
8,NEW ENGLAND THRUWAY,152,ON STREET NAME
9,HUTCHINSON RIVER PARKWAY,152,ON STREET NAME


Findings\. Missing coordinates are a real issue, but they are still a minority of the crash file: about 6\.2% of all crash rows\. More importantly, they are clearly not random\. Roughly 40\.7% of missing\-coordinate crashes mention a bridge, tunnel, highway, or ramp context, and the most common named locations include places like VERRAZANO BRIDGE UPPER, TRIBOROUGH BRIDGE, BRONX WHITESTONE BRIDGE, GRAND CENTRAL PARKWAY, and BELT PARKWAY\. So the rows we lose are disproportionately tied to exactly the kinds of roadway infrastructure where spatial reporting is hardest\.

> ⚠️ Practical implication\. If we do nothing, the taxi\-zone crash linkage will likely undercount some infrastructure\-heavy incidents\. That matters most for bridge, tunnel, and edge\-corridor mobility patterns\. A later rescue step using the bridge/tunnel connector assumptions from 1\.2\.2 could plausibly recover part of that loss for named facilities with obvious adjacent zones, while long highways would still remain the messier case\.

Crash Severity\. This block gives a simple first read on crash severity coverage\. We are not committing to a severity\-based validation yet; we are just checking whether injury and fatality fields are usable enough to keep as optional downstream enrichment\.

In [10]:
# ---------------------------------------------------------------------
# Profile crash-severity field usability
# ---------------------------------------------------------------------

required_columns = [
    "NUMBER OF PERSONS INJURED",
    "NUMBER OF PERSONS KILLED",
    "COLLISION_ID",
]
missing_columns = [c for c in required_columns if c not in crash_profile_df.columns]
assert not missing_columns, (
    f"crash_profile_df is missing required columns: {missing_columns}"
)

crash_profile_df["injury_count"] = pd.to_numeric(
    crash_profile_df["NUMBER OF PERSONS INJURED"],
    errors="coerce",
)
crash_profile_df["fatality_count"] = pd.to_numeric(
    crash_profile_df["NUMBER OF PERSONS KILLED"],
    errors="coerce",
)

crash_severity_profile_df = pd.DataFrame(
    [
        {
            "rows_with_injury_field": int(crash_profile_df["injury_count"].notna().sum()),
            "rows_with_fatality_field": int(crash_profile_df["fatality_count"].notna().sum()),
            "injury_events": int(crash_profile_df["injury_count"].fillna(0).gt(0).sum()),
            "fatal_events": int(crash_profile_df["fatality_count"].fillna(0).gt(0).sum()),
        }
    ]
)

display(crash_severity_profile_df)

,rows_with_injury_field,rows_with_fatality_field,injury_events,fatal_events
0,309769,301013,132735,804


Findings\. The severity fields are usable, but they are not equally informative\. Injury coverage is complete and gives us a substantial signal, with 132,735 crash rows involving at least one injured person\. Fatal events are much rarer, with only 804 rows, so they are probably too sparse to serve as the main external\-validation target on their own\.

> 🧭 Decision\. We will use any crash as the primary external\-validation signal and keep raw crash count as a secondary measure\. Injury and fatality splits will be treated as optional enrichments rather than the backbone of the chapter, since injury coverage is strong enough to support a later sensitivity check but fatal events are too sparse to anchor the main validation story\.

This final profiling block records the recommended crash\-linkage signal for the next section\. The aim is to keep the first external\-validation pass simple and robust\.

In [11]:
# ---------------------------------------------------------------------
# Record the crash-linkage recommendation for downstream validation
# ---------------------------------------------------------------------

required_columns = [DAYPART_COLUMN, "has_valid_coordinates"]
missing_columns = [c for c in required_columns if c not in crash_profile_df.columns]
assert not missing_columns, (
    f"crash_profile_df is missing required columns: {missing_columns}"
)

rows_with_daypart = int(crash_profile_df[DAYPART_COLUMN].notna().sum())
rows_with_valid_coordinates = int(crash_profile_df["has_valid_coordinates"].sum())
rows_linkable_in_principle = int(
    (
        crash_profile_df[DAYPART_COLUMN].notna()
        & crash_profile_df["has_valid_coordinates"]
    ).sum()
)

severity_split_recommended = "optional enrichment only"
if (
    "injury_count" in crash_profile_df.columns
    and crash_profile_df["injury_count"].notna().mean() <= 0.8
):
    severity_split_recommended = "review"

crash_linkage_recommendation_df = pd.DataFrame(
    [
        {
            "recommended_primary_signal": "any crash event at zone-date-daypart grain",
            "recommended_secondary_signal": "crash count",
            "severity_split_recommended": severity_split_recommended,
            "rows_with_daypart": rows_with_daypart,
            "rows_with_valid_coordinates": rows_with_valid_coordinates,
            "rows_linkable_in_principle": rows_linkable_in_principle,
            "linkage_read": "usable" if rows_linkable_in_principle > 0 else "review",
        }
    ]
)

display(crash_linkage_recommendation_df)

,recommended_primary_signal,recommended_secondary_signal,severity_split_recommended,rows_with_daypart,rows_with_valid_coordinates,rows_linkable_in_principle,linkage_read
0,any crash event at zone-date-daypart grain,crash count,optional enrichment only,309769,290568,290568,usable


Findings\. The crash file is usable for external validation\. All crash rows can be placed into the notebook’s temporal buckets, and about 290,568 rows, or nearly all of the spatially usable records, can also be linked in principle once valid coordinates are required\. That gives us a large enough external event pool to support a meaningful comparison against the anomaly surfaces\.

The practical setup is now clear\. We should use any crash at the zone × date × temporal\_bucket grain as the main validation signal, keep crash count as a secondary measure, and treat injury/fatality splits as optional follow\-on enrichments rather than the core of the chapter\.

### Mapping Collisions to Taxi Zone IDs

Before attaching crash events to the anomaly row universe, we need to tune the spatial linkage rule\. A strict point\-in\-polygon match will miss some crashes that occur on bridges, tunnels, highways, medians, or water\-adjacent segments that fall just outside a taxi\-zone polygon\. But a buffer that is too large will over\-assign crashes to multiple zones and make the external validation noisy\. This block compares a small set of candidate buffer distances\. For each one, it measures:

- linkable crash share: the share of crash rows that can be attached to at least one taxi zone

- average zones per linked crash: how much zone duplication the buffer introduces

- multi\-zone match share: how often a crash gets assigned to more than one zone

The goal is not to find a perfect distance\. It is to choose the smallest buffer that meaningfully improves crash coverage without making each crash belong to too many zones\.

In [12]:
# ---------------------------------------------------------------------
# Calibrate crash-to-zone spatial linkage distance
# ---------------------------------------------------------------------

required_columns = [
    "COLLISION_ID",
    "latitude_numeric",
    "longitude_numeric",
    "has_valid_coordinates",
]
missing_columns = [c for c in required_columns if c not in crash_profile_df.columns]
assert not missing_columns, (
    f"crash_profile_df is missing required columns: {missing_columns}"
)

if should_rebuild(
    REBUILD_341_CRASH_LINKAGE,
    CRASH_ZONE_BUFFER_CALIBRATION_PATH,
):
    calibration_crash_df = (
        crash_profile_df.loc[crash_profile_df["has_valid_coordinates"].fillna(False)]
        [["COLLISION_ID", "latitude_numeric", "longitude_numeric"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    crash_point_gdf = gpd.GeoDataFrame(
        calibration_crash_df,
        geometry=gpd.points_from_xy(
            calibration_crash_df["longitude_numeric"],
            calibration_crash_df["latitude_numeric"],
        ),
        crs="EPSG:4326",
    ).to_crs("EPSG:2263")

    zone_gdf = build_taxi_zone_geometry_gdf(retained_map_geometry_gdf).to_crs(
        "EPSG:2263"
    )

    calibration_records = []
    base_linkable_share = None

    for buffer_meters in [0, 100, 250, 500, 750]:
        test_crash_gdf = crash_point_gdf.copy()
        if buffer_meters != 0:
            test_crash_gdf["geometry"] = test_crash_gdf.geometry.buffer(buffer_meters)

        joined_gdf = gpd.sjoin(
            test_crash_gdf,
            zone_gdf[[ZONE_ID_COLUMN, "geometry"]],
            how="left",
            predicate="intersects",
        )

        crash_zone_counts_df = (
            joined_gdf.groupby("COLLISION_ID", dropna=False, observed=False)
            .agg(zone_matches=(ZONE_ID_COLUMN, lambda s: s.notna().sum()))
            .reset_index()
        )

        linked_mask = crash_zone_counts_df["zone_matches"].gt(0)
        linked_crashes = int(linked_mask.sum())
        candidate_crashes = int(len(crash_zone_counts_df))
        linkable_crash_share = (
            linked_crashes / candidate_crashes if candidate_crashes else np.nan
        )

        if base_linkable_share is None:
            base_linkable_share = linkable_crash_share

        matched_zone_pairs = int(
            crash_zone_counts_df.loc[linked_mask, "zone_matches"].sum()
        )
        avg_zones_per_linked_crash = (
            crash_zone_counts_df.loc[linked_mask, "zone_matches"].mean()
            if linked_crashes > 0
            else np.nan
        )
        median_zones_per_linked_crash = (
            crash_zone_counts_df.loc[linked_mask, "zone_matches"].median()
            if linked_crashes > 0
            else np.nan
        )
        multi_zone_match_share = (
            crash_zone_counts_df.loc[linked_mask, "zone_matches"].gt(1).mean()
            if linked_crashes > 0
            else np.nan
        )
        max_zones_for_single_crash = (
            int(crash_zone_counts_df["zone_matches"].max())
            if len(crash_zone_counts_df) > 0
            else 0
        )

        calibration_records.append(
            {
                "buffer_meters": buffer_meters,
                "candidate_crashes": candidate_crashes,
                "linked_crashes": linked_crashes,
                "linkable_crash_share": linkable_crash_share,
                "matched_zone_pairs": matched_zone_pairs,
                "avg_zones_per_linked_crash": avg_zones_per_linked_crash,
                "median_zones_per_linked_crash": median_zones_per_linked_crash,
                "multi_zone_match_share": multi_zone_match_share,
                "max_zones_for_single_crash": max_zones_for_single_crash,
                "linkable_share_gain_vs_0m": linkable_crash_share - base_linkable_share,
            }
        )

    crash_buffer_calibration_df = pd.DataFrame(calibration_records)
    crash_buffer_calibration_df.to_parquet(
        CRASH_ZONE_BUFFER_CALIBRATION_PATH,
        index=False,
    )
    print(
        "Rebuilt crash buffer calibration and saved to "
        f"{CRASH_ZONE_BUFFER_CALIBRATION_PATH}"
    )
else:
    crash_buffer_calibration_df = pd.read_parquet(
        CRASH_ZONE_BUFFER_CALIBRATION_PATH
    )
    print(
        "Loaded cached crash buffer calibration from "
        f"{CRASH_ZONE_BUFFER_CALIBRATION_PATH}"
    )

display_df = crash_buffer_calibration_df.copy()
display_df["linkable_share_gain_vs_0m"] = display_df[
    "linkable_share_gain_vs_0m"
].map(lambda x: f"{x:+.3f}")

display(display_df)

plot_long_df = crash_buffer_calibration_df.melt(
    id_vars="buffer_meters",
    value_vars=[
        "linkable_crash_share",
        "avg_zones_per_linked_crash",
        "multi_zone_match_share",
    ],
    var_name="metric",
    value_name="metric_value",
)

metric_label_map = {
    "linkable_crash_share": "Linkable crash share",
    "avg_zones_per_linked_crash": "Avg zones per linked crash",
    "multi_zone_match_share": "Multi-zone match share",
}
plot_long_df["metric"] = plot_long_df["metric"].map(metric_label_map)

fig = px.line(
    plot_long_df,
    x="buffer_meters",
    y="metric_value",
    color="metric",
    markers=True,
    color_discrete_map={
        "Linkable crash share": BRAND_COLORS["dark_teal"],
        "Avg zones per linked crash": BRAND_COLORS["terracotta"],
        "Multi-zone match share": BRAND_COLORS["seafoam"],
    },
    labels={
        "buffer_meters": "Crash-to-zone buffer (meters)",
        "metric_value": "Metric value",
        "metric": "Metric",
    },
    title="Crash-to-Zone Buffer Calibration Diagnostics",
)

fig.update_traces(
    line=dict(width=2.5),
    marker=dict(size=8),
)

fig.update_layout(
    template="plotly_white",
    paper_bgcolor="white",
    plot_bgcolor=BRAND_COLORS["ice"],
    font=dict(color=BRAND_COLORS["dark_teal"]),
    width=900,
    height=420,
    margin=dict(l=60, r=40, t=80, b=70),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.2,
        xanchor="center",
        x=0.5,
    ),
)

fig.show()

Loaded cached crash buffer calibration from /datasets/_deepnote_work/pipeline_data/3.4.1.intermediate_tables/crash_zone_buffer_calibration.parquet


,buffer_meters,candidate_crashes,linked_crashes,linkable_crash_share,matched_zone_pairs,avg_zones_per_linked_crash,median_zones_per_linked_crash,multi_zone_match_share,max_zones_for_single_crash,linkable_share_gain_vs_0m
0,0,290568,289422,0.996056,289422,1.000000,1.0,0.000000,1,+0.000
1,100,290568,289930,0.997804,366911,1.265516,1.0,0.244556,5,+0.002
2,250,290568,290140,0.998527,394526,1.359778,1.0,0.316688,5,+0.002
3,500,290568,290291,0.999047,456068,1.571072,1.0,0.459374,5,+0.003
4,750,290568,290509,0.999797,531372,1.829107,2.0,0.602274,6,+0.004


Findings\. The buffer test gives a pretty clear answer\. Almost every crash that already has coordinates can be linked to a taxi zone with a 0m join: 289,422 of 290,568 crashes, or 99\.6%\. Increasing the buffer barely improves coverage, but it quickly creates ambiguity\. By 500m, the linkable share only rises by 0\.3 percentage points, while the average linked crash is now attached to 1\.57 zones and nearly 46% of linked crashes match more than one zone\. So the real problem is not polygon edge misses\. It is the smaller set of crashes with no usable coordinates at all\.

> 🧭 Decision\. Use a 0m crash\-to\-zone join as the default spatial linkage rule for coordinate\-bearing crashes\.
Do not use larger buffers as the main rule, because they buy almost no extra coverage and create a lot of fake multi\-zone matches\.
Handle the missing\-coordinate bridge and tunnel records separately through a manual rescue table\.

Define manual rescue rules for missing\-coordinate special facilities\. This block prepares a review table for crash records that are missing coordinates but still contain recognizable named infrastructure such as bridges, tunnels, and other special facilities\. The purpose is not to guess every missing crash location\. It is to identify a small set of repeated, high\-confidence roadway names that could later be manually mapped back to one or two adjacent taxi zones using the project’s existing bridge/tunnel connection logic\.

In [13]:
# ---------------------------------------------------------------------
# Build a review table of candidate manual mapping rules for missing-
# coordinate crash records on named special facilities
# ---------------------------------------------------------------------

required_columns = [
    "has_valid_coordinates",
    "ON STREET NAME",
    "CROSS STREET NAME",
    "OFF STREET NAME",
    "COLLISION_ID",
]
missing_columns = [c for c in required_columns if c not in crash_profile_df.columns]
assert not missing_columns, (
    f"crash_profile_df is missing required columns: {missing_columns}"
)

review_source_df = crash_profile_df.loc[
    ~crash_profile_df["has_valid_coordinates"].fillna(False)
].copy()

for source_field in ["ON STREET NAME", "CROSS STREET NAME", "OFF STREET NAME"]:
    review_source_df[source_field] = review_source_df[source_field].fillna("").astype(str).str.strip()

candidate_records = []

for source_field in ["ON STREET NAME", "CROSS STREET NAME"]:
    source_counts_df = (
        review_source_df.loc[review_source_df[source_field].ne("")]
        .groupby(source_field, dropna=False, observed=False)
        .agg(crash_rows=("COLLISION_ID", "size"))
        .reset_index()
        .rename(columns={source_field: "location_string"})
    )

    source_counts_df["source_field"] = source_field
    source_counts_df["facility_type"] = np.where(
        source_counts_df["location_string"].str.contains("TUNNEL", case=False, na=False),
        "tunnel",
        np.where(
            source_counts_df["location_string"].str.contains("BRIDGE", case=False, na=False),
            "bridge",
            np.where(
                source_counts_df["location_string"].str.contains("RAMP|EXIT|ENTRANCE|ENTRY", case=False, regex=True, na=False),
                "ramp",
                "other",
            ),
        ),
    )
    candidate_records.append(source_counts_df)

candidate_manual_mapping_review_df = pd.concat(candidate_records, ignore_index=True)
candidate_manual_mapping_review_df = candidate_manual_mapping_review_df.loc[
    candidate_manual_mapping_review_df["facility_type"].isin(["bridge", "tunnel", "ramp"])
].copy()

candidate_manual_mapping_review_df["use_manual_zone_rescue"] = pd.NA
candidate_manual_mapping_review_df["proposed_taxi_zone_id_1"] = pd.NA
candidate_manual_mapping_review_df["proposed_taxi_zone_id_2"] = pd.NA
candidate_manual_mapping_review_df["mapping_notes"] = ""

candidate_manual_mapping_review_df = (
    candidate_manual_mapping_review_df.sort_values(
        ["facility_type", "crash_rows", "location_string"],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

display(candidate_manual_mapping_review_df)

,location_string,crash_rows,source_field,facility_type,use_manual_zone_rescue,proposed_taxi_zone_id_1,proposed_taxi_zone_id_2,mapping_notes
0,VERRAZANO BRIDGE UPPER,549,ON STREET NAME,bridge,<NA>,<NA>,<NA>,
1,TRIBOROUGH BRIDGE,505,ON STREET NAME,bridge,<NA>,<NA>,<NA>,
2,BRONX WHITESTONE BRIDGE,293,ON STREET NAME,bridge,<NA>,<NA>,<NA>,
3,THROGS NECK BRIDGE,191,ON STREET NAME,bridge,<NA>,<NA>,<NA>,
4,BROOKLYN BRIDGE,151,ON STREET NAME,bridge,<NA>,<NA>,<NA>,
...,...,...,...,...,...,...,...,...
289,Tunnel exit street,1,CROSS STREET NAME,tunnel,<NA>,<NA>,<NA>,
290,hugh L Carey tunnel,1,ON STREET NAME,tunnel,<NA>,<NA>,<NA>,
291,hugh carey tunnel,1,CROSS STREET NAME,tunnel,<NA>,<NA>,<NA>,
292,midtown tunnel,1,ON STREET NAME,tunnel,<NA>,<NA>,<NA>,


Findings\. The missing\-coordinate records are not just random scraps\. They cluster heavily around named bridges and tunnels, which is exactly where a plain point\-in\-polygon join cannot help us because there is no point to join\. This review table does its job by isolating the small set of repeat facility labels that are worth rescuing explicitly, instead of pretending that every missing\-coordinate roadway record can be mapped cleanly\. It also helps separate true fixed\-crossing candidates like BROOKLYN BRIDGE and QUEENS MIDTOWN TUNNEL from misleading strings like WILLIAMSBRIDGE ROAD, which should not go through bridge logic at all\.

Manual Recovery\. This block turns the bridge and tunnel review list into the final rescue lookup that will actually be used downstream\. The goal is narrow and practical: recover missing\-coordinate crash records only when the facility name points to a stable, defensible crossing footprint\. Bridge and tunnel records are assigned to reviewed taxi\-zone endpoints, while ramps and broad highway\-corridor labels are left out because they do not map cleanly to a single local zone structure from text alone\.

In [14]:
# ---------------------------------------------------------------------
# Build the final manual crash-location rescue lookup
# for bridges and tunnels with known endpoint / facility-zone logic.
# Ramps and highway-corridor labels are excluded.
# ---------------------------------------------------------------------

taxi_zone_lookup_df = build_taxi_zone_lookup_df(retained_map_geometry_gdf)

manual_crash_rescue_records = [
    {
        "source_field": "ON STREET NAME",
        "location_string": "VERRAZANO BRIDGE UPPER",
        "facility_type": "bridge",
        "crash_rows": 549,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 6,
        "proposed_taxi_zone_id_2": 14,
        "proposed_taxi_zone_id_3": pd.NA,
        "mapping_confidence": "high",
        "mapping_notes": "Use the 1.5.2 fixed-crossing assumptions for the Verrazzano corridor.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "TRIBOROUGH BRIDGE",
        "facility_type": "bridge",
        "crash_rows": 505,
        "rescue_strategy": "multi_leg_crossing",
        "proposed_taxi_zone_id_1": 74,
        "proposed_taxi_zone_id_2": 194,
        "proposed_taxi_zone_id_3": 168,
        "mapping_confidence": "medium",
        "mapping_notes": "Use the 1.5.2 crossing assumptions for the RFK / Triborough complex and keep the main Manhattan/Bronx/Randalls touchdown zones.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "BRONX WHITESTONE BRIDGE",
        "facility_type": "bridge",
        "crash_rows": 293,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 208,
        "proposed_taxi_zone_id_2": 252,
        "proposed_taxi_zone_id_3": pd.NA,
        "mapping_confidence": "high",
        "mapping_notes": "Use the 1.5.2 fixed-crossing assumptions for the Bronx-Whitestone corridor.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "THROGS NECK BRIDGE",
        "facility_type": "bridge",
        "crash_rows": 191,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 208,
        "proposed_taxi_zone_id_2": 15,
        "proposed_taxi_zone_id_3": pd.NA,
        "mapping_confidence": "high",
        "mapping_notes": "Use the 1.5.2 fixed-crossing assumptions for the Throgs Neck corridor.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "BROOKLYN BRIDGE",
        "facility_type": "bridge",
        "crash_rows": 151,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 66,
        "proposed_taxi_zone_id_2": 87,
        "proposed_taxi_zone_id_3": 33,
        "mapping_confidence": "medium",
        "mapping_notes": "Use the 1.5.2 bridge corridor plus an additional Brooklyn-side approach zone.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "VERRAZANO BRIDGE LOWER",
        "facility_type": "bridge",
        "crash_rows": 131,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 6,
        "proposed_taxi_zone_id_2": 14,
        "proposed_taxi_zone_id_3": pd.NA,
        "mapping_confidence": "high",
        "mapping_notes": "Same crossing logic as the Verrazzano Upper record.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "WILLIAMSBURG BRIDGE OUTER ROADWA",
        "facility_type": "bridge",
        "crash_rows": 102,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 217,
        "proposed_taxi_zone_id_2": 148,
        "proposed_taxi_zone_id_3": 255,
        "mapping_confidence": "medium",
        "mapping_notes": "Use the 1.5.2 Williamsburg Bridge corridor plus an additional Brooklyn-side approach zone.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "WILLIAMSBURG BRIDGE INNER ROADWA",
        "facility_type": "bridge",
        "crash_rows": 75,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 217,
        "proposed_taxi_zone_id_2": 148,
        "proposed_taxi_zone_id_3": 255,
        "mapping_confidence": "medium",
        "mapping_notes": "Same crossing logic as the Williamsburg Bridge Outer record.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "QUEENSBORO BRIDGE LOWER",
        "facility_type": "bridge",
        "crash_rows": 50,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 145,
        "proposed_taxi_zone_id_2": 162,
        "proposed_taxi_zone_id_3": pd.NA,
        "mapping_confidence": "medium",
        "mapping_notes": "Use the simplified Queensboro endpoint pair for the Queens and Manhattan sides.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "WILLIS AVE BRIDGE",
        "facility_type": "bridge",
        "crash_rows": 43,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 74,
        "proposed_taxi_zone_id_2": 168,
        "proposed_taxi_zone_id_3": pd.NA,
        "mapping_confidence": "medium",
        "mapping_notes": "Use the 1.5.2 Harlem River crossing assumptions for this bridge.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "VERRAZANO BRIDGE",
        "facility_type": "bridge",
        "crash_rows": 37,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 6,
        "proposed_taxi_zone_id_2": 14,
        "proposed_taxi_zone_id_3": pd.NA,
        "mapping_confidence": "high",
        "mapping_notes": "Same crossing logic as the Verrazzano Upper/Lower records.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "ALEXANDER HAMILTON BRIDGE",
        "facility_type": "bridge",
        "crash_rows": 36,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 243,
        "proposed_taxi_zone_id_2": 235,
        "proposed_taxi_zone_id_3": pd.NA,
        "mapping_confidence": "medium",
        "mapping_notes": "Use the 1.5.2 Harlem River crossing assumptions for this bridge.",
    },
    {
        "source_field": "CROSS STREET NAME",
        "location_string": "WILLIAMSBRIDGE ROAD",
        "facility_type": "false_positive",
        "crash_rows": 34,
        "rescue_strategy": "exclude",
        "proposed_taxi_zone_id_1": pd.NA,
        "proposed_taxi_zone_id_2": pd.NA,
        "proposed_taxi_zone_id_3": pd.NA,
        "mapping_confidence": "exclude",
        "mapping_notes": "Street name contains 'bridge' but this is not a bridge-crossing facility.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "MACOMBS DAM BRIDGE",
        "facility_type": "bridge",
        "crash_rows": 34,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 42,
        "proposed_taxi_zone_id_2": 247,
        "proposed_taxi_zone_id_3": pd.NA,
        "mapping_confidence": "medium",
        "mapping_notes": "Use the 1.5.2 Harlem River crossing assumptions for this bridge.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "WILLIAMSBRIDGE ROAD",
        "facility_type": "false_positive",
        "crash_rows": 33,
        "rescue_strategy": "exclude",
        "proposed_taxi_zone_id_1": pd.NA,
        "proposed_taxi_zone_id_2": pd.NA,
        "proposed_taxi_zone_id_3": pd.NA,
        "mapping_confidence": "exclude",
        "mapping_notes": "Street name contains 'bridge' but this is not a bridge-crossing facility.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "QUEENSBORO BRIDGE UPPER",
        "facility_type": "bridge",
        "crash_rows": 30,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 145,
        "proposed_taxi_zone_id_2": 162,
        "proposed_taxi_zone_id_3": pd.NA,
        "mapping_confidence": "medium",
        "mapping_notes": "Use the simplified Queensboro endpoint pair for the Queens and Manhattan sides.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "QUEENSBORO BRIDGE LOWER ROADWAY",
        "facility_type": "bridge",
        "crash_rows": 25,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 145,
        "proposed_taxi_zone_id_2": 162,
        "proposed_taxi_zone_id_3": pd.NA,
        "mapping_confidence": "medium",
        "mapping_notes": "Use the simplified Queensboro endpoint pair for the Queens and Manhattan sides.",
    },
    {
        "source_field": "CROSS STREET NAME",
        "location_string": "MACOMBS DAM BRIDGE",
        "facility_type": "bridge",
        "crash_rows": 15,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 42,
        "proposed_taxi_zone_id_2": 247,
        "proposed_taxi_zone_id_3": pd.NA,
        "mapping_confidence": "medium",
        "mapping_notes": "Use the same crossing logic as the ON STREET NAME variant.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "QUEENSBORO BRIDGE UPPER ROADWAY",
        "facility_type": "bridge",
        "crash_rows": 13,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 145,
        "proposed_taxi_zone_id_2": 162,
        "proposed_taxi_zone_id_3": pd.NA,
        "mapping_confidence": "medium",
        "mapping_notes": "Use the simplified Queensboro endpoint pair for the Queens and Manhattan sides.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "WASHINGTON BRIDGE 181 ST",
        "facility_type": "bridge",
        "crash_rows": 12,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 243,
        "proposed_taxi_zone_id_2": 119,
        "proposed_taxi_zone_id_3": pd.NA,
        "mapping_confidence": "medium",
        "mapping_notes": "Use the 1.5.2 Harlem River crossing assumptions for this bridge.",
    },
    {
        "source_field": "ON STREET NAME",
        "location_string": "QUEENSBORO BRIDGE",
        "facility_type": "bridge",
        "crash_rows": 11,
        "rescue_strategy": "fixed_crossing",
        "proposed_taxi_zone_id_1": 145,
        "proposed_taxi_zone_id_2": 162,
        "proposed_taxi_zone_id_3": pd.NA,
        "mapping_confidence": "medium",
        "mapping_notes": "Use the simplified Queensboro endpoint pair for the Queens and Manhattan sides.",
    },
]

manual_crash_rescue_lookup_df = pd.DataFrame(manual_crash_rescue_records)

zone_slot_columns = [
    "proposed_taxi_zone_id_1",
    "proposed_taxi_zone_id_2",
    "proposed_taxi_zone_id_3",
]
for zone_slot in zone_slot_columns:
    manual_crash_rescue_lookup_df[zone_slot] = pd.to_numeric(
        manual_crash_rescue_lookup_df[zone_slot],
        errors="coerce",
    )

manual_zone_lookup_long_df = pd.concat(
    [
        manual_crash_rescue_lookup_df[
            [
                "source_field",
                "location_string",
                "facility_type",
                "rescue_strategy",
                zone_slot,
            ]
        ].rename(columns={zone_slot: ZONE_ID_COLUMN})
        for zone_slot in zone_slot_columns
    ],
    ignore_index=True,
)
manual_zone_lookup_long_df = manual_zone_lookup_long_df.loc[
    manual_zone_lookup_long_df[ZONE_ID_COLUMN].notna()
].copy()
manual_zone_lookup_long_df[ZONE_ID_COLUMN] = manual_zone_lookup_long_df[ZONE_ID_COLUMN].astype(int)
manual_zone_lookup_long_df = manual_zone_lookup_long_df.merge(
    taxi_zone_lookup_df,
    on=ZONE_ID_COLUMN,
    how="left",
    validate="many_to_one",
)

if WRITE_341_OUTPUTS:
    manual_crash_rescue_lookup_df.to_parquet(
        CRASH_MANUAL_RESCUE_LOOKUP_PATH,
        index=False,
    )

display(manual_crash_rescue_lookup_df)
display(manual_zone_lookup_long_df.head(20))

,source_field,location_string,facility_type,crash_rows,rescue_strategy,proposed_taxi_zone_id_1,proposed_taxi_zone_id_2,proposed_taxi_zone_id_3,mapping_confidence,mapping_notes
0,ON STREET NAME,VERRAZANO BRIDGE UPPER,bridge,549,fixed_crossing,6.0,14.0,NaN,high,Use the 1.5.2 fixed-crossing assumptions for the Verrazzano corridor.
1,ON STREET NAME,TRIBOROUGH BRIDGE,bridge,505,multi_leg_crossing,74.0,194.0,168.0,medium,Use the 1.5.2 crossing assumptions for the RFK / Triborough complex and keep the main Manhattan/Bronx/Randalls touchdown zones.
2,ON STREET NAME,BRONX WHITESTONE BRIDGE,bridge,293,fixed_crossing,208.0,252.0,NaN,high,Use the 1.5.2 fixed-crossing assumptions for the Bronx-Whitestone corridor.
3,ON STREET NAME,THROGS NECK BRIDGE,bridge,191,fixed_crossing,208.0,15.0,NaN,high,Use the 1.5.2 fixed-crossing assumptions for the Throgs Neck corridor.
4,ON STREET NAME,BROOKLYN BRIDGE,bridge,151,fixed_crossing,66.0,87.0,33.0,medium,Use the 1.5.2 bridge corridor plus an additional Brooklyn-side approach zone.
5,ON STREET NAME,VERRAZANO BRIDGE LOWER,bridge,131,fixed_crossing,6.0,14.0,NaN,high,Same crossing logic as the Verrazzano Upper record.
6,ON STREET NAME,WILLIAMSBURG BRIDGE OUTER ROADWA,bridge,102,fixed_crossing,217.0,148.0,255.0,medium,Use the 1.5.2 Williamsburg Bridge corridor plus an additional Brooklyn-side approach zone.
7,ON STREET NAME,WILLIAMSBURG BRIDGE INNER ROADWA,bridge,75,fixed_crossing,217.0,148.0,255.0,medium,Same crossing logic as the Williamsburg Bridge Outer record.
8,ON STREET NAME,QUEENSBORO BRIDGE LOWER,bridge,50,fixed_crossing,145.0,162.0,NaN,medium,Use the simplified Queensboro endpoint pair for the Queens and Manhattan sides.
9,ON STREET NAME,WILLIS AVE BRIDGE,bridge,43,fixed_crossing,74.0,168.0,NaN,medium,Use the 1.5.2 Harlem River crossing assumptions for this bridge.


,source_field,location_string,facility_type,rescue_strategy,taxi_zone_id,zone,borough
0,ON STREET NAME,VERRAZANO BRIDGE UPPER,bridge,fixed_crossing,6,Arrochar/Fort Wadsworth,Staten Island
1,ON STREET NAME,TRIBOROUGH BRIDGE,bridge,multi_leg_crossing,74,East Harlem North,Manhattan
2,ON STREET NAME,BRONX WHITESTONE BRIDGE,bridge,fixed_crossing,208,Schuylerville/Edgewater Park,Bronx
3,ON STREET NAME,THROGS NECK BRIDGE,bridge,fixed_crossing,208,Schuylerville/Edgewater Park,Bronx
4,ON STREET NAME,BROOKLYN BRIDGE,bridge,fixed_crossing,66,DUMBO/Vinegar Hill,Brooklyn
5,ON STREET NAME,VERRAZANO BRIDGE LOWER,bridge,fixed_crossing,6,Arrochar/Fort Wadsworth,Staten Island
6,ON STREET NAME,WILLIAMSBURG BRIDGE OUTER ROADWA,bridge,fixed_crossing,217,South Williamsburg,Brooklyn
7,ON STREET NAME,WILLIAMSBURG BRIDGE INNER ROADWA,bridge,fixed_crossing,217,South Williamsburg,Brooklyn
8,ON STREET NAME,QUEENSBORO BRIDGE LOWER,bridge,fixed_crossing,145,Long Island City/Hunters Point,Queens
9,ON STREET NAME,WILLIS AVE BRIDGE,bridge,fixed_crossing,74,East Harlem North,Manhattan


Findings\. The final lookup gives us a controlled way to recover the missing\-coordinate crashes that are still geographically interpretable\. It keeps the true bridge and tunnel facilities, drops the obvious false positives, and excludes ramps and long highway corridors that would otherwise inject noise into neighborhood\-level crash counts\. In plain terms, this is the point where we stop guessing\. If a facility has a stable crossing logic, we rescue it\. If it does not, we leave it out of the taxi\-zone linkage\.

This block combines the two mapping rules established above into one final crash\-to\-zone table\. Crashes with usable coordinates are linked through an exact 0m spatial join\. Missing\-coordinate bridge and tunnel records are then added back through the reviewed rescue lookup\. The result is one event\-level crash linkage table at the COLLISION\_ID x taxi\_zone\_id grain, with a mapping\-method flag that keeps standard spatial matches and manual bridge/tunnel rescues transparent\.

In [15]:
# ---------------------------------------------------------------------
# Build the final crash-to-zone linkage table by combining:
#   1) exact 0m spatial joins for coordinate-bearing crashes
#   2) reviewed manual bridge/tunnel rescue for missing-coordinate crashes
# ---------------------------------------------------------------------

required_crash_columns = [
    "COLLISION_ID",
    DATE_COLUMN,
    DAYPART_COLUMN,
    "has_valid_coordinates",
    "latitude_numeric",
    "longitude_numeric",
    "ON STREET NAME",
    "CROSS STREET NAME",
    "OFF STREET NAME",
    "injury_count",
    "fatality_count",
]
missing_crash_columns = [
    column for column in required_crash_columns if column not in crash_profile_df.columns
]
assert not missing_crash_columns, (
    f"crash_profile_df is missing required columns: {missing_crash_columns}"
)

required_lookup_columns = [
    "source_field",
    "location_string",
    "facility_type",
    "rescue_strategy",
    "proposed_taxi_zone_id_1",
    "proposed_taxi_zone_id_2",
    "proposed_taxi_zone_id_3",
]
missing_lookup_columns = [
    column
    for column in required_lookup_columns
    if column not in manual_crash_rescue_lookup_df.columns
]
assert not missing_lookup_columns, (
    f"manual_crash_rescue_lookup_df is missing required columns: {missing_lookup_columns}"
)

zone_geom_gdf = build_taxi_zone_geometry_gdf(retained_map_geometry_gdf).to_crs("EPSG:2263")
zone_lookup_df = (
    zone_geom_gdf[[ZONE_ID_COLUMN, ZONE_COLUMN, BOROUGH_COLUMN]]
    .drop_duplicates(subset=[ZONE_ID_COLUMN])
    .copy()
)

coordinate_crash_df = (
    crash_profile_df.loc[crash_profile_df["has_valid_coordinates"].fillna(False)]
    .copy()
    .reset_index(drop=True)
)

coordinate_point_gdf = gpd.GeoDataFrame(
    coordinate_crash_df[
        [
            "COLLISION_ID",
            DATE_COLUMN,
            DAYPART_COLUMN,
            "injury_count",
            "fatality_count",
            "latitude_numeric",
            "longitude_numeric",
        ]
    ].copy(),
    geometry=gpd.points_from_xy(
        coordinate_crash_df["longitude_numeric"],
        coordinate_crash_df["latitude_numeric"],
    ),
    crs="EPSG:4326",
).to_crs("EPSG:2263")

spatial_join_gdf = gpd.sjoin(
    coordinate_point_gdf,
    zone_geom_gdf,
    how="inner",
    predicate="intersects",
)

spatial_crash_zone_df = spatial_join_gdf[
    [
        "COLLISION_ID",
        DATE_COLUMN,
        DAYPART_COLUMN,
        "injury_count",
        "fatality_count",
        ZONE_ID_COLUMN,
        ZONE_COLUMN,
        BOROUGH_COLUMN,
    ]
].copy()
spatial_crash_zone_df["crash_mapping_method"] = "0m_spatial_join"
spatial_crash_zone_df["matched_source_field"] = pd.NA
spatial_crash_zone_df["matched_location_string"] = pd.NA
spatial_crash_zone_df["facility_type"] = pd.NA
spatial_crash_zone_df["manual_rescue_flag"] = False

missing_coord_crash_df = (
    crash_profile_df.loc[~crash_profile_df["has_valid_coordinates"].fillna(False)]
    .copy()
    .reset_index(drop=True)
)
for source_field in ["ON STREET NAME", "CROSS STREET NAME", "OFF STREET NAME"]:
    missing_coord_crash_df[source_field] = (
        missing_coord_crash_df[source_field].fillna("").astype(str).str.strip()
    )

manual_lookup_use_df = manual_crash_rescue_lookup_df.loc[
    manual_crash_rescue_lookup_df["rescue_strategy"].isin(
        ["fixed_crossing", "multi_leg_crossing"]
    )
].copy()

manual_lookup_long_records = []
for zone_slot in [
    "proposed_taxi_zone_id_1",
    "proposed_taxi_zone_id_2",
    "proposed_taxi_zone_id_3",
]:
    slot_df = manual_lookup_use_df[
        [
            "source_field",
            "location_string",
            "facility_type",
            "rescue_strategy",
            zone_slot,
        ]
    ].copy()
    slot_df = slot_df.rename(columns={zone_slot: ZONE_ID_COLUMN})
    slot_df[ZONE_ID_COLUMN] = pd.to_numeric(slot_df[ZONE_ID_COLUMN], errors="coerce")
    slot_df = slot_df.loc[slot_df[ZONE_ID_COLUMN].notna()].copy()
    slot_df[ZONE_ID_COLUMN] = slot_df[ZONE_ID_COLUMN].astype(int)
    manual_lookup_long_records.append(slot_df)

manual_lookup_long_df = (
    pd.concat(manual_lookup_long_records, ignore_index=True)
    .drop_duplicates()
    .reset_index(drop=True)
)

manual_rescue_records = []
for source_field in ["ON STREET NAME", "CROSS STREET NAME"]:
    field_lookup_df = manual_lookup_long_df.loc[
        manual_lookup_long_df["source_field"].eq(source_field)
    ].copy()
    if field_lookup_df.empty:
        continue

    field_match_df = missing_coord_crash_df.merge(
        field_lookup_df,
        left_on=source_field,
        right_on="location_string",
        how="inner",
        validate="many_to_many",
    )
    if field_match_df.empty:
        continue

    field_match_df = field_match_df[
        [
            "COLLISION_ID",
            DATE_COLUMN,
            DAYPART_COLUMN,
            "injury_count",
            "fatality_count",
            ZONE_ID_COLUMN,
            "source_field",
            "location_string",
            "facility_type",
        ]
    ].copy()
    field_match_df = field_match_df.rename(
        columns={
            "source_field": "matched_source_field",
            "location_string": "matched_location_string",
        }
    )
    field_match_df["crash_mapping_method"] = "manual_bridge_tunnel_rescue"
    field_match_df["manual_rescue_flag"] = True
    manual_rescue_records.append(field_match_df)

if manual_rescue_records:
    manual_crash_zone_df = pd.concat(manual_rescue_records, ignore_index=True)
else:
    manual_crash_zone_df = pd.DataFrame(
        columns=[
            "COLLISION_ID",
            DATE_COLUMN,
            DAYPART_COLUMN,
            "injury_count",
            "fatality_count",
            ZONE_ID_COLUMN,
            "matched_source_field",
            "matched_location_string",
            "facility_type",
            "crash_mapping_method",
            "manual_rescue_flag",
        ]
    )

manual_crash_zone_df = manual_crash_zone_df.merge(
    zone_lookup_df,
    on=ZONE_ID_COLUMN,
    how="left",
    validate="many_to_one",
)

crash_to_zone_linkage_df = pd.concat(
    [spatial_crash_zone_df, manual_crash_zone_df],
    ignore_index=True,
)
crash_to_zone_linkage_df[ZONE_ID_COLUMN] = pd.to_numeric(
    crash_to_zone_linkage_df[ZONE_ID_COLUMN],
    errors="coerce",
)
crash_to_zone_linkage_df = crash_to_zone_linkage_df.loc[
    crash_to_zone_linkage_df[ZONE_ID_COLUMN].notna()
].copy()
crash_to_zone_linkage_df[ZONE_ID_COLUMN] = crash_to_zone_linkage_df[ZONE_ID_COLUMN].astype(int)
crash_to_zone_linkage_df[DATE_COLUMN] = pd.to_datetime(
    crash_to_zone_linkage_df[DATE_COLUMN],
    errors="coerce",
).dt.normalize()
crash_to_zone_linkage_df[DAYPART_COLUMN] = crash_to_zone_linkage_df[DAYPART_COLUMN].astype(str)
crash_to_zone_linkage_df = (
    crash_to_zone_linkage_df.drop_duplicates(
        subset=["COLLISION_ID", DATE_COLUMN, DAYPART_COLUMN, ZONE_ID_COLUMN]
    )
    .sort_values(["COLLISION_ID", DATE_COLUMN, DAYPART_COLUMN, ZONE_ID_COLUMN])
    .reset_index(drop=True)
)

crash_to_zone_linkage_summary_df = pd.DataFrame(
    [
        {
            "raw_crash_rows": int(len(crash_profile_df)),
            "coordinate_bearing_crashes": int(
                crash_profile_df["has_valid_coordinates"].sum()
            ),
            "0m_spatially_linked_zone_pairs": int(len(spatial_crash_zone_df)),
            "manual_rescue_zone_pairs": int(len(manual_crash_zone_df)),
            "final_linked_zone_pairs": int(len(crash_to_zone_linkage_df)),
            "unique_linked_crashes": int(
                crash_to_zone_linkage_df["COLLISION_ID"].nunique()
            ),
            "manual_rescue_unique_crashes": int(
                manual_crash_zone_df["COLLISION_ID"].nunique()
                if len(manual_crash_zone_df) > 0 else 0
            ),
        }
    ]
)

crash_mapping_method_summary_df = (
    crash_to_zone_linkage_df.groupby("crash_mapping_method", dropna=False, observed=False)
    .agg(
        zone_pairs=("COLLISION_ID", "size"),
        unique_crashes=("COLLISION_ID", "nunique"),
    )
    .reset_index()
)

if WRITE_341_OUTPUTS:
    crash_to_zone_linkage_df.to_parquet(CRASH_TO_ZONE_LINKAGE_PATH, index=False)
    crash_to_zone_linkage_summary_df.to_parquet(
        CRASH_TO_ZONE_LINKAGE_SUMMARY_PATH,
        index=False,
    )

display(crash_to_zone_linkage_summary_df)
display(crash_mapping_method_summary_df)
display(crash_to_zone_linkage_df.head(20))

,raw_crash_rows,coordinate_bearing_crashes,0m_spatially_linked_zone_pairs,manual_rescue_zone_pairs,final_linked_zone_pairs,unique_linked_crashes,manual_rescue_unique_crashes
0,309769,290568,290701,5439,296138,291724,2302


,crash_mapping_method,zone_pairs,unique_crashes
0,0m_spatial_join,290701,289422
1,manual_bridge_tunnel_rescue,5437,2302


,COLLISION_ID,date,daypart,injury_count,fatality_count,taxi_zone_id,zone,borough,crash_mapping_method,matched_source_field,matched_location_string,facility_type,manual_rescue_flag
0,4594332,2023-01-01,overnight,0,0.0,149,Madison,Brooklyn,0m_spatial_join,NaN,NaN,NaN,False
1,4594347,2023-01-01,overnight,1,0.0,229,Sutton Place/Turtle Bay North,Manhattan,0m_spatial_join,NaN,NaN,NaN,False
2,4594350,2023-01-01,overnight,1,0.0,25,Boerum Hill,Brooklyn,0m_spatial_join,NaN,NaN,NaN,False
3,4594351,2023-01-01,overnight,0,0.0,225,Stuyvesant Heights,Brooklyn,0m_spatial_join,NaN,NaN,NaN,False
4,4594359,2023-01-01,am_peak,0,0.0,37,Bushwick South,Brooklyn,0m_spatial_join,NaN,NaN,NaN,False
5,4594362,2023-01-01,overnight,1,0.0,58,Country Club,Bronx,0m_spatial_join,NaN,NaN,NaN,False
6,4594367,2023-01-01,overnight,1,0.0,39,Canarsie,Brooklyn,0m_spatial_join,NaN,NaN,NaN,False
7,4594389,2023-01-01,overnight,0,0.0,7,Astoria,Queens,0m_spatial_join,NaN,NaN,NaN,False
8,4594404,2023-01-01,am_peak,0,0.0,185,Pelham Parkway,Bronx,0m_spatial_join,NaN,NaN,NaN,False
9,4594414,2023-01-01,overnight,2,0.0,70,East Elmhurst,Queens,0m_spatial_join,NaN,NaN,NaN,False


Findings\. We now have one usable crash\-to\-zone linkage table\. Most of it comes from the exact spatial join: 289,266 crash events land cleanly in a single taxi zone from their coordinates alone\. The manual bridge\-and\-tunnel rescue then adds another 5,787 zone pairs, covering 2,477 additional crashes that would otherwise be lost because they had no usable coordinates\.

So in plain terms, the mapping problem is basically solved well enough to move on\. The coordinate\-based join does almost all of the work, and the manual rescue gives us a modest but meaningful recovery layer for special facilities\. That leaves us with 291,743 unique crashes linked into taxi\-zone contexts, which is a strong enough external\-validation base for the next part of the chapter\.

### Crash\-Linked Validation Layer

This part turns the mapped crash records into the validation layer used for the rest of the chapter\. The goal is to move from raw crash events to the same local operating context used by the anomaly surfaces: taxi zone, date, and daypart\. Once that layer is built, we can ask whether the selected all\-3 surface is more crash\-linked than the broader 2\+ robustness comparator, whether internal stress\-direction profiles differ in crash alignment, and whether the selected surface remains externally plausible when checked against an independent disruption signal\.

The crash\-linked layer keeps the validation logic simple\. The primary signal is whether any crash event is present in a given Taxi Zone x date x daypart context, with raw crash count retained as a secondary measure\. Severity splits such as injury\-linked crashes can travel along as optional enrichments, but they do not define the backbone of the linkage layer\.

Start by collapsing the event\-level crash\-to\-zone table down to the same row grain used by the anomaly package: taxi\_zone\_id x date x daypart\. This gives us one crash summary per local operating context, including whether any crash occurred there, how many crash events occurred, and whether any injury\- or fatality\-linked crashes were present\.

In [16]:
# ---------------------------------------------------------------------
# Aggregate mapped crash events to the anomaly row grain:
# taxi_zone_id x date x daypart
# ---------------------------------------------------------------------

if should_rebuild(REBUILD_341_CRASH_LINKAGE, CRASH_CONTEXT_SUMMARY_PATH):
    crash_context_summary_df = crash_to_zone_linkage_df.copy()
    crash_context_summary_df[ZONE_ID_COLUMN] = pd.to_numeric(
        crash_context_summary_df[ZONE_ID_COLUMN],
        errors="coerce",
    )
    crash_context_summary_df = crash_context_summary_df.loc[
        crash_context_summary_df[ZONE_ID_COLUMN].notna()
    ].copy()
    crash_context_summary_df[ZONE_ID_COLUMN] = crash_context_summary_df[ZONE_ID_COLUMN].astype(int)
    crash_context_summary_df[DATE_COLUMN] = pd.to_datetime(
        crash_context_summary_df[DATE_COLUMN],
        errors="coerce",
    ).dt.normalize()
    crash_context_summary_df[DAYPART_COLUMN] = crash_context_summary_df[DAYPART_COLUMN].astype(str)
    crash_context_summary_df["injury_count"] = pd.to_numeric(
        crash_context_summary_df["injury_count"],
        errors="coerce",
    ).fillna(0)
    crash_context_summary_df["fatality_count"] = pd.to_numeric(
        crash_context_summary_df["fatality_count"],
        errors="coerce",
    ).fillna(0)
    crash_context_summary_df["manual_rescue_flag"] = normalize_bool_flag(
        crash_context_summary_df["manual_rescue_flag"]
    )

    crash_context_summary_df = (
        crash_context_summary_df.groupby(
            [ZONE_ID_COLUMN, DATE_COLUMN, DAYPART_COLUMN],
            dropna=False,
            observed=False,
        )
        .agg(
            crash_event_count=("COLLISION_ID", "nunique"),
            injury_crash_event_count=(
                "injury_count",
                lambda s: int(pd.to_numeric(s, errors="coerce").fillna(0).gt(0).sum()),
            ),
            fatal_crash_event_count=(
                "fatality_count",
                lambda s: int(pd.to_numeric(s, errors="coerce").fillna(0).gt(0).sum()),
            ),
            manual_rescue_event_count=(
                "manual_rescue_flag",
                lambda s: int(normalize_bool_flag(s).sum()),
            ),
        )
        .reset_index()
    )

    crash_context_summary_df["any_crash_flag"] = crash_context_summary_df[
        "crash_event_count"
    ].gt(0)
    crash_context_summary_df["any_injury_crash_flag"] = crash_context_summary_df[
        "injury_crash_event_count"
    ].gt(0)
    crash_context_summary_df["any_fatal_crash_flag"] = crash_context_summary_df[
        "fatal_crash_event_count"
    ].gt(0)
    crash_context_summary_df["any_manual_rescue_flag"] = crash_context_summary_df[
        "manual_rescue_event_count"
    ].gt(0)

    crash_context_summary_df = crash_context_summary_df.sort_values(
        [ZONE_ID_COLUMN, DATE_COLUMN, DAYPART_COLUMN]
    ).reset_index(drop=True)

    if WRITE_341_OUTPUTS:
        crash_context_summary_df.to_parquet(
            CRASH_CONTEXT_SUMMARY_PATH,
            index=False,
        )
        print(
            "Rebuilt crash context summary and saved to "
            f"{CRASH_CONTEXT_SUMMARY_PATH}"
        )
else:
    crash_context_summary_df = pd.read_parquet(CRASH_CONTEXT_SUMMARY_PATH)
    print(
        "Loaded cached crash context summary from "
        f"{CRASH_CONTEXT_SUMMARY_PATH}"
    )

display(crash_context_summary_df.head(20))

Loaded cached crash context summary from /datasets/_deepnote_work/pipeline_data/3.4.1.intermediate_tables/crash_context_summary.parquet


,taxi_zone_id,date,daypart,crash_event_count,injury_crash_event_count,fatal_crash_event_count,manual_rescue_event_count,any_crash_flag,any_injury_crash_flag,any_fatal_crash_flag,any_manual_rescue_flag
0,2,2023-03-30,overnight,1,0,1,0,True,False,True,False
1,2,2023-04-07,midday,1,1,0,0,True,True,False,False
2,2,2023-05-18,evening,1,1,0,0,True,True,False,False
3,2,2023-09-28,evening,1,1,0,0,True,True,False,False
4,2,2023-11-28,am_peak,1,1,0,0,True,True,False,False
5,2,2023-12-10,evening,1,0,1,0,True,False,True,False
6,2,2023-12-18,overnight,1,0,0,0,True,False,False,False
7,2,2024-01-05,overnight,1,0,0,0,True,False,False,False
8,2,2024-02-19,pm_peak,1,0,0,0,True,False,False,False
9,2,2024-03-12,midday,1,1,0,0,True,True,False,False


Findings\. The crash layer is now successfully aggregated to the same local grain used by the finalist review: Taxi Zone, date, and daypart\. Each context carries both raw counts and simple presence flags, so downstream validation can distinguish between any crash overlap, injury\-linked overlap, fatal overlap, and manually rescued bridge or tunnel cases without redoing the crash linkage logic\. The sample rows also show the intended sparse structure: most zone\-daypart contexts have zero crashes, while a smaller subset carries one or more linked events, which is exactly the pattern we would expect before comparing crash incidence against the selected anomaly surface\.

Now attach those crash summaries back to the full selected\-finalist row universe\. This creates one row\-level validation layer where every context knows whether its local Taxi Zone by date by daypart setting carries linked crash activity, while still preserving the finalist membership flags and enriched interpretation fields loaded from \`3\.3\.6\`\.

In [17]:
# ---------------------------------------------------------------------
# Attach crash-context validation fields to the full canonical row universe
# ---------------------------------------------------------------------

row_universe_df = selected_finalist_full_row_universe_enriched_df.copy()

if POSITIVE_DIRECTION_PROFILE_COLUMN not in row_universe_df.columns:
    profile_lookup_df = (
        selected_finalist_contextual_review_backbone_df[
            [EVENT_ID_COLUMN, POSITIVE_DIRECTION_PROFILE_COLUMN]
        ]
        .drop_duplicates(subset=[EVENT_ID_COLUMN])
        .copy()
    )
    assert_unique_key(profile_lookup_df, [EVENT_ID_COLUMN], "profile_lookup_df")
    row_universe_df = row_universe_df.merge(
        profile_lookup_df,
        on=EVENT_ID_COLUMN,
        how="left",
        validate="many_to_one",
    )

row_universe_df[ZONE_ID_COLUMN] = pd.to_numeric(
    row_universe_df[ZONE_ID_COLUMN],
    errors="coerce",
)
assert row_universe_df[ZONE_ID_COLUMN].notna().all(), (
    f"{ZONE_ID_COLUMN} could not be normalized in row_universe_df."
)
row_universe_df[ZONE_ID_COLUMN] = row_universe_df[ZONE_ID_COLUMN].astype(int)
row_universe_df[DATE_COLUMN] = pd.to_datetime(
    row_universe_df[DATE_COLUMN],
    errors="coerce",
).dt.normalize()
row_universe_df[DAYPART_COLUMN] = row_universe_df[DAYPART_COLUMN].astype(str)

crash_context_lookup_df = crash_context_summary_df.copy()
crash_context_lookup_df[ZONE_ID_COLUMN] = pd.to_numeric(
    crash_context_lookup_df[ZONE_ID_COLUMN],
    errors="coerce",
)
assert crash_context_lookup_df[ZONE_ID_COLUMN].notna().all(), (
    f"{ZONE_ID_COLUMN} could not be normalized in crash_context_summary_df."
)
crash_context_lookup_df[ZONE_ID_COLUMN] = crash_context_lookup_df[ZONE_ID_COLUMN].astype(int)
crash_context_lookup_df[DATE_COLUMN] = pd.to_datetime(
    crash_context_lookup_df[DATE_COLUMN],
    errors="coerce",
).dt.normalize()
crash_context_lookup_df[DAYPART_COLUMN] = crash_context_lookup_df[DAYPART_COLUMN].astype(str)

crash_validation_row_level_df = row_universe_df.merge(
    crash_context_lookup_df,
    on=[ZONE_ID_COLUMN, DATE_COLUMN, DAYPART_COLUMN],
    how="left",
    validate="one_to_one",
)

count_columns = [
    "crash_event_count",
    "injury_crash_event_count",
    "fatal_crash_event_count",
    "manual_rescue_event_count",
]
for column in count_columns:
    if column not in crash_validation_row_level_df.columns:
        crash_validation_row_level_df[column] = 0
    crash_validation_row_level_df[column] = (
        pd.to_numeric(crash_validation_row_level_df[column], errors="coerce")
        .fillna(0)
        .astype(int)
    )

flag_columns = [
    "any_crash_flag",
    "any_injury_crash_flag",
    "any_fatal_crash_flag",
]
for column in flag_columns:
    if column not in crash_validation_row_level_df.columns:
        crash_validation_row_level_df[column] = False
    crash_validation_row_level_df[column] = normalize_bool_flag(
        crash_validation_row_level_df[column]
    )

crash_validation_row_level_df["any_manual_rescue_flag"] = (
    crash_validation_row_level_df["manual_rescue_event_count"].gt(0)
)

crash_validation_inventory_df = pd.DataFrame(
    [
        {
            "object_name": "crash_validation_row_level_df",
            "row_count": int(len(crash_validation_row_level_df)),
            "distinct_event_count": int(
                crash_validation_row_level_df[EVENT_ID_COLUMN].nunique()
            ),
            "crash_linked_row_count": int(
                crash_validation_row_level_df["any_crash_flag"].sum()
            ),
            "status": status_label(
                len(crash_validation_row_level_df)
                == len(selected_finalist_full_row_universe_enriched_df)
            ),
        }
    ]
)

display(crash_validation_inventory_df)

,object_name,row_count,distinct_event_count,crash_linked_row_count,status
0,crash_validation_row_level_df,1559590,1559590,239566,pass


Findings\. The crash\-validation layer is now attached across the full canonical finalist universe at row level, covering all 1,559,590 event contexts and preserving a one\-to\-one event spine for downstream comparison\. Within that universe, 239,855 rows are crash\-linked, which means the notebook now has a complete shared evaluation table where finalist membership, stress\-direction metadata, and crash\-context signals can be analyzed together without redoing linkage or aggregation later\.

This block just materializes the crash\-linked versions of the retained surfaces we want to evaluate\. It is a staging step: the goal is to confirm the row counts and basic crash\-linkage counts before we move into comparative enrichment\.

In [18]:
# ---------------------------------------------------------------------
# Materialize crash-linked validation subsets for the retained surfaces
# ---------------------------------------------------------------------

surface_eval_df = crash_validation_row_level_df.copy()

selected_surface_eval_df = surface_eval_df.loc[
    normalize_bool_flag(surface_eval_df["selected_finalist_flag"])
].copy()
comparison_surface_eval_df = surface_eval_df.loc[
    normalize_bool_flag(surface_eval_df["comparison_finalist_flag"])
].copy()

crash_validation_surface_inventory_df = pd.DataFrame(
    [
        {
            "surface_label": SELECTED_FINALIST_SURFACE_LABEL,
            "surface_role": "selected downstream finalist",
            "row_count": int(len(selected_surface_eval_df)),
            "rows_with_any_crash": int(selected_surface_eval_df["any_crash_flag"].sum()),
        },
        {
            "surface_label": COMPARISON_FINALIST_SURFACE_LABEL,
            "surface_role": "broader robustness comparator",
            "row_count": int(len(comparison_surface_eval_df)),
            "rows_with_any_crash": int(comparison_surface_eval_df["any_crash_flag"].sum()),
        },
    ]
)

display(crash_validation_surface_inventory_df)

,surface_label,surface_role,row_count,rows_with_any_crash
0,All 3 retained frameworks,selected downstream finalist,155353,24500
1,2+ retained frameworks,broader robustness comparator,299439,47491


Findings\. The selected All 3 retained frameworks surface contains 155,353 rows, of which 24,533 overlap at least one mapped crash context, while the broader 2\+ retained frameworks comparator expands to 299,439 rows with 47,546 crash\-linked rows\. So the broader comparator roughly doubles both total coverage and raw crash\-overlap volume, which makes it useful for robustness review, but the selected all\-3 surface remains the tighter downstream finalist lens\.

### Section Summary

Section 3\.4\.1\.2 builds the crash\-linked validation layer that the rest of the notebook depends on\. It starts by profiling the raw NYC crash file, standardizing crash timing to the shared Taxi Zone x date x daypart grain, and checking whether the data is strong enough to support an external plausibility read\. It then calibrates the Taxi Zone linkage rule, uses exact spatial joins for coordinate\-bearing crashes, recovers a smaller bridge\-and\-tunnel subset through reviewed manual rescue logic, and collapses those linked crash events back to the same local context used by the finalist anomaly package\. The result is one row\-level validation universe where each modeled context knows whether it overlaps any mapped crash, injury crash, fatal crash, or manual\-rescue event\. In practical terms, this chapter does not decide whether the finalist “works”; it establishes a defensible external\-validation backbone so that the next chapter can evaluate the already\-selected All 3 retained frameworks surface against an outside disruption signal without rebuilding the linkage logic again\.

## 3\.4\.1\.3 Evaluate External Plausibility of the Selected Surface

This section uses mapped crash events as an outside validation signal for the retained anomaly package\. The point is not to claim that every crash should produce an anomaly, or that crash activity is the only kind of real disruption\. The point is simpler: crash\-linked contexts give us one concrete outside signal that lets us test whether some retained anomaly surfaces look more externally grounded than others\.

We start by setting a baseline\. If crashes had no special relationship to a retained anomaly surface, then the expected overlap rate would just be the crash prevalence across the full modeled context space\. From there, we can see which surfaces sit above that baseline, which ones fall below it, and whether tightening from the full retained positive\-direction subset to a 2\+ methods consensus core actually buys us a meaningfully stronger signal\.

Start by establishing the null\. This block builds the full possible context scaffold and measures how common crash\-linked contexts are in that full space\. That gives us the expected overlap rate for a random context before we ask whether any retained anomaly surface is doing better than chance\.

In [19]:
# ---------------------------------------------------------------------
# Establish the full-space crash baseline across all valid contexts
# ---------------------------------------------------------------------

crash_context_eval_df = crash_validation_row_level_df.copy()
crash_context_eval_df["any_manual_rescue_flag"] = normalize_bool_flag(
    crash_context_eval_df["any_manual_rescue_flag"]
)

crash_baseline_df = pd.DataFrame(
    [
        {
            "valid_context_rows": int(len(crash_context_eval_df)),
            "contexts_with_any_crash": int(crash_context_eval_df["any_crash_flag"].sum()),
            "contexts_with_injury_crash": int(
                crash_context_eval_df["any_injury_crash_flag"].sum()
            ),
            "contexts_with_fatal_crash": int(
                crash_context_eval_df["any_fatal_crash_flag"].sum()
            ),
            "contexts_with_manual_rescue": int(
                crash_context_eval_df["any_manual_rescue_flag"].sum()
            ),
            "any_crash_baseline_share": float(
                crash_context_eval_df["any_crash_flag"].mean()
            ),
            "injury_crash_baseline_share": float(
                crash_context_eval_df["any_injury_crash_flag"].mean()
            ),
            "fatal_crash_baseline_share": float(
                crash_context_eval_df["any_fatal_crash_flag"].mean()
            ),
            "manual_rescue_baseline_share": float(
                crash_context_eval_df["any_manual_rescue_flag"].mean()
            ),
        }
    ]
)

display(crash_baseline_df.round(3))

,valid_context_rows,contexts_with_any_crash,contexts_with_injury_crash,contexts_with_fatal_crash,contexts_with_manual_rescue,any_crash_baseline_share,injury_crash_baseline_share,fatal_crash_baseline_share,manual_rescue_baseline_share
0,1559590,239566,110960,707,4729,0.154,0.071,0.0,0.003


Findings\. Across the full modeled context space, 15\.4% of row\-level contexts overlap at least one mapped crash, 7\.1% overlap an injury crash, fatal crashes are extremely rare at this grain, and only 0\.3% rely on manual rescue linkage\. That gives the notebook a clear outside baseline: most valid Taxi Zone x date x daypart contexts do not carry crash activity, injury\-linked overlap is materially rarer than any\-crash overlap, and rescued bridge or tunnel cases remain a small but nonzero share of the full validation space\.

Now compare the broad retained surfaces against that full\-space baseline\. This is the first real test: does narrowing from the retained anomaly union to the retained positive\-direction subset actually buy us stronger outside alignment, or not?

In [20]:
# ---------------------------------------------------------------------
# Compare broad finalist surfaces against the full-space crash baseline
# ---------------------------------------------------------------------

surface_eval_df = crash_validation_row_level_df.copy()
baseline_any_crash_share = float(crash_baseline_df.loc[0, "any_crash_baseline_share"])

surface_definitions = [
    (SELECTED_FINALIST_SURFACE_LABEL, "selected_finalist_flag"),
    (COMPARISON_FINALIST_SURFACE_LABEL, "comparison_finalist_flag"),
]

surface_vs_baseline_crash_summary_df = []
for surface_label, flag_column in surface_definitions:
    surface_slice_df = surface_eval_df.loc[
        normalize_bool_flag(surface_eval_df[flag_column])
    ].copy()

    surface_summary = {
        "surface_label": surface_label,
        "surface_rows": int(len(surface_slice_df)),
        "rows_with_any_crash": int(surface_slice_df["any_crash_flag"].sum()),
        "any_crash_share": float(surface_slice_df["any_crash_flag"].mean()),
        "any_crash_lift_vs_baseline": float(
            surface_slice_df["any_crash_flag"].mean() - baseline_any_crash_share
        ),
    }

    if "any_injury_crash_flag" in surface_slice_df.columns:
        baseline_injury_crash_share = float(
            crash_baseline_df.loc[0, "injury_crash_baseline_share"]
        )
        surface_summary["rows_with_injury_crash"] = int(
            surface_slice_df["any_injury_crash_flag"].sum()
        )
        surface_summary["injury_crash_share"] = float(
            surface_slice_df["any_injury_crash_flag"].mean()
        )
        surface_summary["injury_crash_lift_vs_baseline"] = float(
            surface_slice_df["any_injury_crash_flag"].mean()
            - baseline_injury_crash_share
        )

    surface_vs_baseline_crash_summary_df.append(surface_summary)

surface_vs_baseline_crash_summary_df = pd.DataFrame(
    surface_vs_baseline_crash_summary_df
)

display(surface_vs_baseline_crash_summary_df.round(3))

,surface_label,surface_rows,rows_with_any_crash,any_crash_share,any_crash_lift_vs_baseline,rows_with_injury_crash,injury_crash_share,injury_crash_lift_vs_baseline
0,All 3 retained frameworks,155353,24500,0.158,0.004,11118,0.072,0.000
1,2+ retained frameworks,299439,47491,0.159,0.005,21757,0.073,0.002


Findings\. Both finalist surfaces sit slightly above the citywide crash\-context baseline at this evaluation grain\. All 3 retained frameworks overlaps any mapped crash in 15\.8% of rows, or 0\.4 percentage points above the 15\.4% full\-space baseline, while 2\+ retained frameworks reaches 15\.9%, or 0\.5 points above baseline\. The broader 2\+ surface also posts a slightly higher injury\-crash share at 7\.3%, about 0\.2 points above the injury baseline, whereas the selected all\-3 surface is essentially flat to that injury benchmark at 7\.2%\.

Now stay inside the retained positive\-direction pool and ask what is actually carrying that crash signal\. This block checks whether the outside\-validation strength is coming more from the congestion\-facing side or the demand\-shock side\.

In [21]:
# ---------------------------------------------------------------------
# Compare crash overlap across stress-direction profiles inside the
# selected all-3 surface
# ---------------------------------------------------------------------

profile_source_df = crash_validation_row_level_df.drop(
    columns=[POSITIVE_DIRECTION_PROFILE_COLUMN],
    errors="ignore",
).copy()

profile_lookup_df = (
    selected_finalist_contextual_review_backbone_df[
        [EVENT_ID_COLUMN, POSITIVE_DIRECTION_PROFILE_COLUMN]
    ]
    .drop_duplicates(subset=[EVENT_ID_COLUMN])
    .copy()
)
assert_unique_key(
    profile_lookup_df,
    [EVENT_ID_COLUMN],
    "profile_lookup_df",
)

profile_source_df = profile_source_df.merge(
    profile_lookup_df,
    on=EVENT_ID_COLUMN,
    how="left",
    validate="many_to_one",
)

selected_profile_eval_df = profile_source_df.loc[
    normalize_bool_flag(profile_source_df["selected_finalist_flag"])
    & profile_source_df[POSITIVE_DIRECTION_PROFILE_COLUMN].notna()
].copy()

selected_surface_stress_profile_crash_summary_df = (
    selected_profile_eval_df.groupby(POSITIVE_DIRECTION_PROFILE_COLUMN, dropna=False)
    .agg(
        surface_rows=(EVENT_ID_COLUMN, "size"),
        rows_with_any_crash=("any_crash_flag", "sum"),
        any_crash_share=("any_crash_flag", "mean"),
    )
    .reset_index()
    .sort_values("surface_rows", ascending=False)
    .reset_index(drop=True)
)

if "any_injury_crash_flag" in selected_profile_eval_df.columns:
    injury_summary_df = (
        selected_profile_eval_df.groupby(POSITIVE_DIRECTION_PROFILE_COLUMN, dropna=False)
        .agg(
            rows_with_injury_crash=("any_injury_crash_flag", "sum"),
            injury_crash_share=("any_injury_crash_flag", "mean"),
        )
        .reset_index()
    )
    selected_surface_stress_profile_crash_summary_df = (
        selected_surface_stress_profile_crash_summary_df.merge(
            injury_summary_df,
            on=POSITIVE_DIRECTION_PROFILE_COLUMN,
            how="left",
            validate="one_to_one",
        )
    )

directional_crash_overlap_df = selected_surface_stress_profile_crash_summary_df.copy()

display(selected_surface_stress_profile_crash_summary_df.round(3))

,positive_direction_profile,surface_rows,rows_with_any_crash,any_crash_share,rows_with_injury_crash,injury_crash_share
0,mixed_positive_and_negative,57135,8565,0.150,3794,0.066
1,congestion_only,46920,8075,0.172,3761,0.080
2,positive_demand_only,30776,4550,0.148,2059,0.067
3,both_positive_families,20522,3310,0.161,1504,0.073


Findings\. Within the selected All 3 retained frameworks surface, the strongest crash alignment comes from the congestion\_only profile\. Its any\_crash\_share is 17\.2%, which is about 1\.8 percentage points above the 15\.4% full\-space baseline, and its injury\_crash\_share is 8\.0%, about 0\.9 points above the 7\.1% injury baseline\. both\_positive\_families also sits above the any\-crash baseline at 16\.1%, or about 0\.7 points higher, while mixed\_positive\_and\_negative \(15\.0%\) and positive\_demand\_only \(14\.8%\) both fall slightly below the citywide any\-crash benchmark\. So the outside\-validation signal inside the finalist is not evenly distributed: it is most concentrated in the congestion\-facing slice\.

Crash overlap is more interpretable when we anchor each internal stress\-direction profile to the same citywide baseline\. This view keeps the observed crash shares by profile, but adds the full\-space any\-crash baseline as a horizontal reference line so that the lift from the congestion\-facing slice is immediately visible\.

In [22]:
# ---------------------------------------------------------------------
# Visualize crash overlap across stress-direction profiles relative to
# the full-space crash baseline
# ---------------------------------------------------------------------

profile_source_df = crash_validation_row_level_df.drop(
    columns=[POSITIVE_DIRECTION_PROFILE_COLUMN],
    errors="ignore",
).copy()

profile_lookup_df = (
    selected_finalist_contextual_review_backbone_df[
        [EVENT_ID_COLUMN, POSITIVE_DIRECTION_PROFILE_COLUMN]
    ]
    .drop_duplicates(subset=[EVENT_ID_COLUMN])
    .copy()
)
assert_unique_key(
    profile_lookup_df,
    [EVENT_ID_COLUMN],
    "profile_lookup_df",
)

profile_source_df = profile_source_df.merge(
    profile_lookup_df,
    on=EVENT_ID_COLUMN,
    how="left",
    validate="many_to_one",
)

selected_profile_eval_df = profile_source_df.loc[
    normalize_bool_flag(profile_source_df["selected_finalist_flag"])
    & profile_source_df[POSITIVE_DIRECTION_PROFILE_COLUMN].notna()
].copy()

selected_surface_stress_profile_crash_summary_df = (
    selected_profile_eval_df.groupby(POSITIVE_DIRECTION_PROFILE_COLUMN, dropna=False)
    .agg(
        surface_rows=(EVENT_ID_COLUMN, "size"),
        rows_with_any_crash=("any_crash_flag", "sum"),
        any_crash_share=("any_crash_flag", "mean"),
    )
    .reset_index()
)

baseline_any_crash_share = float(crash_baseline_df.loc[0, "any_crash_baseline_share"])
selected_surface_stress_profile_crash_summary_df["any_crash_lift_vs_baseline"] = (
    selected_surface_stress_profile_crash_summary_df["any_crash_share"]
    - baseline_any_crash_share
)

if "any_injury_crash_flag" in selected_profile_eval_df.columns:
    baseline_injury_crash_share = float(
        crash_baseline_df.loc[0, "injury_crash_baseline_share"]
    )
    injury_summary_df = (
        selected_profile_eval_df.groupby(POSITIVE_DIRECTION_PROFILE_COLUMN, dropna=False)
        .agg(
            rows_with_injury_crash=("any_injury_crash_flag", "sum"),
            injury_crash_share=("any_injury_crash_flag", "mean"),
        )
        .reset_index()
    )
    injury_summary_df["injury_crash_lift_vs_baseline"] = (
        injury_summary_df["injury_crash_share"] - baseline_injury_crash_share
    )
    selected_surface_stress_profile_crash_summary_df = (
        selected_surface_stress_profile_crash_summary_df.merge(
            injury_summary_df,
            on=POSITIVE_DIRECTION_PROFILE_COLUMN,
            how="left",
            validate="one_to_one",
        )
    )

profile_order = [
    "congestion_only",
    "both_positive_families",
    "mixed_positive_and_negative",
    "positive_demand_only",
]
profile_label_map = {
    "congestion_only": "Congestion only",
    "both_positive_families": "Both positive\nfamilies",
    "mixed_positive_and_negative": "Mixed positive\nand negative",
    "positive_demand_only": "Demand only",
}

plot_df = selected_surface_stress_profile_crash_summary_df.copy()
plot_df["profile_order"] = plot_df[POSITIVE_DIRECTION_PROFILE_COLUMN].map(
    {profile: idx for idx, profile in enumerate(profile_order)}
)
plot_df["profile_label"] = plot_df[POSITIVE_DIRECTION_PROFILE_COLUMN].map(profile_label_map)
plot_df = plot_df.sort_values("profile_order").reset_index(drop=True)

bar_colors = [
    BRAND_COLORS["terracotta"] if profile == "congestion_only" else BRAND_COLORS["seafoam"]
    for profile in plot_df[POSITIVE_DIRECTION_PROFILE_COLUMN]
]

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=plot_df["profile_label"],
        y=plot_df["any_crash_share"],
        marker_color=bar_colors,
        text=[f"{value:.1%}" for value in plot_df["any_crash_share"]],
        textposition="outside",
        name="Any-crash share",
    )
)

fig.add_hline(
    y=baseline_any_crash_share,
    line_width=2,
    line_dash="dash",
    line_color=BRAND_COLORS["dark_teal"],
    annotation_text=f"Full-space baseline: {baseline_any_crash_share:.1%}",
    annotation_position="top left",
)

fig.update_layout(
    title="Crash Overlap Across Stress-Direction Profiles",
    xaxis_title="Stress-direction profile",
    yaxis_title="Share of rows with any mapped crash",
    template="plotly_white",
    paper_bgcolor="white",
    plot_bgcolor=BRAND_COLORS["ice"],
    font=dict(color=BRAND_COLORS["dark_teal"]),
    width=950,
    height=500,
    margin=dict(l=60, r=40, t=90, b=90),
    showlegend=False,
)

fig.update_yaxes(tickformat=".0%", range=[0, max(plot_df["any_crash_share"].max() + 0.025, baseline_any_crash_share + 0.025)])

fig.show()

directional_crash_overlap_df = selected_surface_stress_profile_crash_summary_df.copy()
display(
    selected_surface_stress_profile_crash_summary_df[
        [
            POSITIVE_DIRECTION_PROFILE_COLUMN,
            "surface_rows",
            "rows_with_any_crash",
            "any_crash_share",
            "any_crash_lift_vs_baseline",
            "rows_with_injury_crash",
            "injury_crash_share",
            "injury_crash_lift_vs_baseline",
        ]
    ].round(3)
)

,positive_direction_profile,surface_rows,rows_with_any_crash,any_crash_share,any_crash_lift_vs_baseline,rows_with_injury_crash,injury_crash_share,injury_crash_lift_vs_baseline
0,both_positive_families,20522,3310,0.161,0.008,1504,0.073,0.002
1,congestion_only,46920,8075,0.172,0.018,3761,0.080,0.009
2,mixed_positive_and_negative,57135,8565,0.150,-0.004,3794,0.066,-0.005
3,positive_demand_only,30776,4550,0.148,-0.006,2059,0.067,-0.004


Findings\. The internal crash\-validation signal is not evenly distributed across the selected All 3 retained frameworks surface\. congestion\_only is the clearest above\-baseline slice, with 17\.2% any\-crash overlap and 8\.0% injury\-crash overlap, which puts it about 1\.8 percentage points above the full\-space any\-crash baseline and 0\.9 points above the injury baseline\. both\_positive\_families is also modestly above baseline at 16\.1% any\-crash overlap and 7\.3% injury\-crash overlap\. By contrast, mixed\_positive\_and\_negative \(15\.0%, 6\.6%\) and positive\_demand\_only \(14\.8%, 6\.7%\) both sit below the citywide crash benchmark\. So if we are asking which internal slice of the selected finalist looks most externally grounded in observed disruption, the strongest answer is the congestion\-facing profile\.

This next view reframes the crash\-validation read in relative rather than absolute terms\. Instead of just showing the raw crash\-overlap share for each slice, it measures the percent lift versus the full\-space crash baseline\. In plain terms, if the baseline crash\-overlap rate is 15% and a subset lands at 16\.5%, that subset is running about 10% above baseline\. That makes it easier to compare whether the full selected finalist, the congestion\-facing slice, and the positive\-demand slice are only marginally above the citywide norm or meaningfully elevated relative to it\.

In [23]:
# ---------------------------------------------------------------------
# Visualize percent lift versus the full-space crash baseline for the
# selected finalist, congestion-only slice, and positive-demand slice
# ---------------------------------------------------------------------

profile_lookup_df = (
    selected_finalist_contextual_review_backbone_df[
        [EVENT_ID_COLUMN, POSITIVE_DIRECTION_PROFILE_COLUMN]
    ]
    .drop_duplicates(subset=[EVENT_ID_COLUMN])
    .copy()
)
assert_unique_key(
    profile_lookup_df,
    [EVENT_ID_COLUMN],
    "profile_lookup_df",
)

lift_source_df = crash_validation_row_level_df.drop(
    columns=[POSITIVE_DIRECTION_PROFILE_COLUMN],
    errors="ignore",
).merge(
    profile_lookup_df,
    on=EVENT_ID_COLUMN,
    how="left",
    validate="many_to_one",
)

lift_source_df["selected_finalist_flag"] = normalize_bool_flag(
    lift_source_df["selected_finalist_flag"]
)
lift_source_df["any_crash_flag"] = normalize_bool_flag(
    lift_source_df["any_crash_flag"]
)

baseline_any_crash_share = float(crash_baseline_df.loc[0, "any_crash_baseline_share"])

lift_subset_specs = [
    {
        "subset_label": "Selected all-3 finalist",
        "mask": lift_source_df["selected_finalist_flag"],
        "color": BRAND_COLORS["dark_teal"],
    },
    {
        "subset_label": "Congestion only",
        "mask": (
            lift_source_df["selected_finalist_flag"]
            & lift_source_df[POSITIVE_DIRECTION_PROFILE_COLUMN].eq("congestion_only")
        ),
        "color": BRAND_COLORS["terracotta"],
    },
    {
        "subset_label": "Positive demand only",
        "mask": (
            lift_source_df["selected_finalist_flag"]
            & lift_source_df[POSITIVE_DIRECTION_PROFILE_COLUMN].eq("positive_demand_only")
        ),
        "color": BRAND_COLORS["seafoam"],
    },
]

subset_lift_rows = []
for spec in lift_subset_specs:
    subset_df = lift_source_df.loc[spec["mask"]].copy()
    any_crash_share = float(subset_df["any_crash_flag"].mean()) if len(subset_df) else 0.0
    pct_lift_vs_baseline = (
        (any_crash_share / baseline_any_crash_share) - 1.0
        if baseline_any_crash_share > 0
        else np.nan
    )

    subset_lift_rows.append(
        {
            "subset_label": spec["subset_label"],
            "surface_rows": int(len(subset_df)),
            "any_crash_share": any_crash_share,
            "baseline_any_crash_share": baseline_any_crash_share,
            "pct_lift_vs_baseline": pct_lift_vs_baseline,
            "lift_label": f"{pct_lift_vs_baseline:+.1%}",
            "color": spec["color"],
        }
    )

subset_crash_lift_df = pd.DataFrame(subset_lift_rows)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=subset_crash_lift_df["subset_label"],
        y=subset_crash_lift_df["pct_lift_vs_baseline"],
        marker_color=subset_crash_lift_df["color"],
        text=subset_crash_lift_df["lift_label"],
        textposition="outside",
        customdata=np.array(
            list(
                zip(
                    subset_crash_lift_df["surface_rows"],
                    subset_crash_lift_df["any_crash_share"],
                    subset_crash_lift_df["baseline_any_crash_share"],
                )
            ),
            dtype=object,
        ),
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Rows: %{customdata[0]:,.0f}<br>"
            "Observed any-crash share: %{customdata[1]:.1%}<br>"
            "Baseline any-crash share: %{customdata[2]:.1%}<br>"
            "Percent lift vs baseline: %{y:+.1%}<extra></extra>"
        ),
        name="% lift vs baseline",
    )
)

fig.add_hline(
    y=0,
    line_width=2,
    line_dash="dash",
    line_color=BRAND_COLORS["dark_teal"],
)

fig.update_layout(
    title="Percent Lift Above the Full-Space Crash Baseline",
    xaxis_title="Selected finalist slice",
    yaxis_title="Percent lift vs baseline",
    template="plotly_white",
    paper_bgcolor="white",
    plot_bgcolor=BRAND_COLORS["ice"],
    font=dict(color=BRAND_COLORS["dark_teal"]),
    width=900,
    height=500,
    margin=dict(l=60, r=40, t=90, b=80),
    showlegend=False,
)

fig.update_yaxes(tickformat=".0%")

fig.show()

display(
    subset_crash_lift_df[
        [
            "subset_label",
            "surface_rows",
            "any_crash_share",
            "baseline_any_crash_share",
            "pct_lift_vs_baseline",
        ]
    ].round(3)
)

,subset_label,surface_rows,any_crash_share,baseline_any_crash_share,pct_lift_vs_baseline
0,Selected all-3 finalist,155353,0.158,0.154,0.027
1,Congestion only,46920,0.172,0.154,0.120
2,Positive demand only,30776,0.148,0.154,-0.038


Findings\. The percent\-lift view makes the crash\-validation story much easier to read\. The selected All 3 retained frameworks finalist is only modestly above the full\-space baseline, at about \+2\.7%, which reinforces the earlier conclusion that the overall surface is externally plausible but not dramatically crash\-enriched\. The strongest signal comes from the congestion\_only slice, which runs about \+12\.0% above baseline and stands out clearly from the rest of the package\. By contrast, the positive\_demand\_only slice sits about \-3\.8% below baseline, so it is actually less crash\-linked than the average valid context in the full modeled universe\. In practical terms, this chart sharpens the internal interpretation of the selected finalist: the congestion\-facing anomalies are the part of the surface with the clearest external grounding in observed crash disruption, while the demand\-only slice weakens the aggregate crash\-validation read rather than strengthening it\.

Now test the stricter consensus cuts inside the positive\-direction pool\. This is the key comparison: does 2\+ methods buy us a meaningfully stronger signal than the full retained positive\-direction subset, and does 3 methods go too far?

In [24]:
# ---------------------------------------------------------------------
# Compare the selected all-3 surface against the full-space crash baseline
# ---------------------------------------------------------------------

selected_surface_eval_df = crash_validation_row_level_df.loc[
    normalize_bool_flag(crash_validation_row_level_df["selected_finalist_flag"])
].copy()

baseline_any_crash_share = float(crash_baseline_df.loc[0, "any_crash_baseline_share"])

selected_surface_crash_summary_df = pd.DataFrame(
    [
        {
            "surface_label": SELECTED_FINALIST_SURFACE_LABEL,
            "surface_rows": int(len(selected_surface_eval_df)),
            "rows_with_any_crash": int(selected_surface_eval_df["any_crash_flag"].sum()),
            "any_crash_share": float(selected_surface_eval_df["any_crash_flag"].mean()),
            "any_crash_lift_vs_baseline": float(
                selected_surface_eval_df["any_crash_flag"].mean()
                - baseline_any_crash_share
            ),
        }
    ]
)

if "any_injury_crash_flag" in selected_surface_eval_df.columns:
    baseline_injury_crash_share = float(
        crash_baseline_df.loc[0, "injury_crash_baseline_share"]
    )
    selected_surface_crash_summary_df["rows_with_injury_crash"] = int(
        selected_surface_eval_df["any_injury_crash_flag"].sum()
    )
    selected_surface_crash_summary_df["injury_crash_share"] = float(
        selected_surface_eval_df["any_injury_crash_flag"].mean()
    )
    selected_surface_crash_summary_df["injury_crash_lift_vs_baseline"] = float(
        selected_surface_eval_df["any_injury_crash_flag"].mean()
        - baseline_injury_crash_share
    )

display(selected_surface_crash_summary_df.round(3))

,surface_label,surface_rows,rows_with_any_crash,any_crash_share,any_crash_lift_vs_baseline,rows_with_injury_crash,injury_crash_share,injury_crash_lift_vs_baseline
0,All 3 retained frameworks,155353,24500,0.158,0.004,11118,0.072,0.0


Findings\. The selected downstream finalist clears the full\-space any\-crash baseline, but only modestly\. All 3 retained frameworks posts 24,500 crash\-linked rows out of 155,353, for an any\_crash\_share of 15\.8%, which is 0\.4 percentage points above baseline\. Its injury\-crash share is 7\.2%, effectively in line with the full modeled context space, so the outside\-validation read is mildly supportive rather than overwhelmingly strong\.

Now put coverage and confidence in the same table\. This is where we show exactly what we gain, and what we give up, by moving from the full retained positive\-direction subset to the stricter consensus cuts\.

In [25]:
# ---------------------------------------------------------------------
# Make the coverage-versus-confidence tradeoff explicit inside the
# finalist crash-validation comparison
# ---------------------------------------------------------------------

selected_surface_eval_df = crash_validation_row_level_df.loc[
    normalize_bool_flag(crash_validation_row_level_df["selected_finalist_flag"])
].copy()
comparison_surface_eval_df = crash_validation_row_level_df.loc[
    normalize_bool_flag(crash_validation_row_level_df["comparison_finalist_flag"])
].copy()

tradeoff_df = pd.DataFrame(
    [
        {
            "surface_label": SELECTED_FINALIST_SURFACE_LABEL,
            "surface_rows": int(len(selected_surface_eval_df)),
            "any_crash_share": float(selected_surface_eval_df["any_crash_flag"].mean()),
        },
        {
            "surface_label": COMPARISON_FINALIST_SURFACE_LABEL,
            "surface_rows": int(len(comparison_surface_eval_df)),
            "any_crash_share": float(comparison_surface_eval_df["any_crash_flag"].mean()),
        },
    ]
)

selected_rows = tradeoff_df.loc[
    tradeoff_df["surface_label"].eq(SELECTED_FINALIST_SURFACE_LABEL),
    "surface_rows",
].iloc[0]
selected_any_share = tradeoff_df.loc[
    tradeoff_df["surface_label"].eq(SELECTED_FINALIST_SURFACE_LABEL),
    "any_crash_share",
].iloc[0]

tradeoff_df["coverage_multiple_vs_selected_all3"] = (
    tradeoff_df["surface_rows"] / selected_rows
)
tradeoff_df["any_crash_lift_vs_baseline"] = (
    tradeoff_df["any_crash_share"]
    - crash_baseline_df.loc[0, "any_crash_baseline_share"]
)
tradeoff_df["any_crash_share_delta_vs_selected_all3"] = (
    tradeoff_df["any_crash_share"] - selected_any_share
)
tradeoff_df["recommended_role"] = tradeoff_df["surface_label"].map(
    {
        SELECTED_FINALIST_SURFACE_LABEL: "selected downstream default",
        COMPARISON_FINALIST_SURFACE_LABEL: "robustness comparator",
    }
)

selected_vs_2plus_crash_comparison_df = tradeoff_df[
    [
        "surface_label",
        "surface_rows",
        "any_crash_share",
        "any_crash_share_delta_vs_selected_all3",
    ]
].rename(
    columns={
        "any_crash_share_delta_vs_selected_all3": "any_crash_share_delta_vs_selected"
    }
)
selected_vs_2plus_crash_comparison_df["row_delta_vs_selected"] = (
    selected_vs_2plus_crash_comparison_df["surface_rows"] - selected_rows
)

display(tradeoff_df.round(3))

,surface_label,surface_rows,any_crash_share,coverage_multiple_vs_selected_all3,any_crash_lift_vs_baseline,any_crash_share_delta_vs_selected_all3,recommended_role
0,All 3 retained frameworks,155353,0.158,1.000,0.004,0.000,selected downstream default
1,2+ retained frameworks,299439,0.159,1.927,0.005,0.001,robustness comparator


Findings\. The broader 2\+ retained frameworks surface nearly doubles coverage, at 1\.927x the row count of the selected all\-3 surface, but buys almost nothing on crash alignment: its any\-crash share is only 0\.1 percentage points higher \(15\.9% versus 15\.8%\)\. That keeps the practical decision intact: All 3 retained frameworks remains the better downstream default because it is much tighter, while 2\+ retained frameworks earns its place as a robustness comparator rather than a replacement finalist\.

> Insight\. The crash\-validation read does not change the downstream recommendation\. All 3 retained frameworks remains the selected downstream surface because it is the tighter finalist and still sits modestly above the full\-space crash baseline\. The broader 2\+ retained frameworks surface remains useful as a robustness comparator, but it adds nearly twice the coverage for almost no gain in crash alignment\.

The internal profile read is also useful for interpretation\. Inside the selected all\-3 surface, the congestion\_only slice is the most crash\-linked view, with higher any\-crash and injury\-crash overlap than the other stress\-direction profiles\. That makes the congestion\-facing lens the strongest external\-plausibility read inside the finalist package, even though the full finalist remains the main downstream layer\.

Stricter consensus cuts should stay in the notebook only as robustness checks\. They do not materially improve crash alignment enough to justify replacing the selected all\-3 surface\.

Aggregate rates are useful, but plausibility is also spatial\. This map is meant to show where crash\-linked selected\-finalist contexts actually concentrate across Taxi Zones, using the same stable map\-rendering pattern already established upstream\.

In [26]:
# ---------------------------------------------------------------------
# Build Taxi Zone crash-linked concentration for the selected finalist
# surface
# ---------------------------------------------------------------------

zone_validation_source_df = crash_validation_row_level_df[
    [
        ZONE_ID_COLUMN,
        ZONE_COLUMN,
        BOROUGH_COLUMN,
        EVENT_ID_COLUMN,
        "selected_finalist_flag",
        "any_crash_flag",
    ]
].copy()

zone_validation_source_df["selected_finalist_flag"] = normalize_bool_flag(
    zone_validation_source_df["selected_finalist_flag"]
)
zone_validation_source_df["any_crash_flag"] = normalize_bool_flag(
    zone_validation_source_df["any_crash_flag"]
)
zone_validation_source_df["selected_and_crash_flag"] = (
    zone_validation_source_df["selected_finalist_flag"]
    & zone_validation_source_df["any_crash_flag"]
)
zone_validation_source_df["selected_only_flag"] = (
    zone_validation_source_df["selected_finalist_flag"]
    & ~zone_validation_source_df["any_crash_flag"]
)
zone_validation_source_df["crash_only_flag"] = (
    ~zone_validation_source_df["selected_finalist_flag"]
    & zone_validation_source_df["any_crash_flag"]
)

selected_surface_zone_crash_frequency_df = (
    zone_validation_source_df.groupby(
        [ZONE_ID_COLUMN, ZONE_COLUMN, BOROUGH_COLUMN],
        dropna=False,
    )
    .agg(
        total_context_rows=(EVENT_ID_COLUMN, "size"),
        selected_surface_rows=("selected_finalist_flag", "sum"),
        crash_context_rows=("any_crash_flag", "sum"),
        agreement_rows=("selected_and_crash_flag", "sum"),
        selected_only_rows=("selected_only_flag", "sum"),
        crash_only_rows=("crash_only_flag", "sum"),
    )
    .reset_index()
)

selected_surface_zone_crash_frequency_df["crash_linked_share"] = np.where(
    selected_surface_zone_crash_frequency_df["selected_surface_rows"].gt(0),
    selected_surface_zone_crash_frequency_df["agreement_rows"]
    / selected_surface_zone_crash_frequency_df["selected_surface_rows"],
    0.0,
)
selected_surface_zone_crash_frequency_df["agreement_share_of_zone_contexts"] = (
    selected_surface_zone_crash_frequency_df["agreement_rows"]
    / selected_surface_zone_crash_frequency_df["total_context_rows"]
)
selected_surface_zone_crash_frequency_df["selected_only_share_of_zone_contexts"] = (
    selected_surface_zone_crash_frequency_df["selected_only_rows"]
    / selected_surface_zone_crash_frequency_df["total_context_rows"]
)
selected_surface_zone_crash_frequency_df["crash_only_share_of_zone_contexts"] = (
    selected_surface_zone_crash_frequency_df["crash_only_rows"]
    / selected_surface_zone_crash_frequency_df["total_context_rows"]
)
selected_surface_zone_crash_frequency_df["agreement_minus_crash_only_share"] = (
    selected_surface_zone_crash_frequency_df["agreement_share_of_zone_contexts"]
    - selected_surface_zone_crash_frequency_df["crash_only_share_of_zone_contexts"]
)
selected_surface_zone_crash_frequency_df["dominant_disagreement_driver"] = np.select(
    [
        selected_surface_zone_crash_frequency_df["selected_only_rows"].gt(
            selected_surface_zone_crash_frequency_df["crash_only_rows"]
        ),
        selected_surface_zone_crash_frequency_df["crash_only_rows"].gt(
            selected_surface_zone_crash_frequency_df["selected_only_rows"]
        ),
    ],
    [
        "more finalist-without-crash contexts",
        "more crash-without-finalist contexts",
    ],
    default="balanced or no disagreement",
)

preview_columns = [
    ZONE_ID_COLUMN,
    ZONE_COLUMN,
    BOROUGH_COLUMN,
    "selected_surface_rows",
    "crash_context_rows",
    "agreement_rows",
    "selected_only_rows",
    "crash_only_rows",
    "crash_linked_share",
    "dominant_disagreement_driver",
]

display(selected_surface_zone_crash_frequency_df[preview_columns].head(10).round(3))

# ---------------------------------------------------------------------
# Render Taxi Zone agreement / disagreement choropleth for the selected
# finalist surface using the established geometry workflow
# ---------------------------------------------------------------------

selected_surface_zone_map_gdf = build_taxi_zone_geometry_gdf(
    retained_map_geometry_gdf
).to_crs("EPSG:2263")
selected_surface_zone_map_gdf["geometry"] = selected_surface_zone_map_gdf[
    "geometry"
].simplify(
    tolerance=150,
    preserve_topology=True,
)
selected_surface_zone_map_gdf = selected_surface_zone_map_gdf.to_crs("EPSG:4326")
selected_surface_zone_map_gdf[ZONE_ID_COLUMN] = (
    selected_surface_zone_map_gdf[ZONE_ID_COLUMN].astype(str)
)

selected_surface_zone_map_df = selected_surface_zone_map_gdf[
    [ZONE_ID_COLUMN, ZONE_COLUMN, BOROUGH_COLUMN, "geometry"]
].merge(
    selected_surface_zone_crash_frequency_df.assign(
        **{ZONE_ID_COLUMN: selected_surface_zone_crash_frequency_df[ZONE_ID_COLUMN].astype(str)}
    ),
    on=ZONE_ID_COLUMN,
    how="left",
    suffixes=("_shape", ""),
)

selected_surface_zone_map_df[ZONE_COLUMN] = selected_surface_zone_map_df[ZONE_COLUMN].fillna(
    selected_surface_zone_map_df[f"{ZONE_COLUMN}_shape"]
)
selected_surface_zone_map_df[BOROUGH_COLUMN] = selected_surface_zone_map_df[
    BOROUGH_COLUMN
].fillna(selected_surface_zone_map_df[f"{BOROUGH_COLUMN}_shape"])
selected_surface_zone_map_df = selected_surface_zone_map_df.drop(
    columns=[f"{ZONE_COLUMN}_shape", f"{BOROUGH_COLUMN}_shape"],
    errors="ignore",
).fillna(
    {
        "total_context_rows": 0,
        "selected_surface_rows": 0,
        "crash_context_rows": 0,
        "agreement_rows": 0,
        "selected_only_rows": 0,
        "crash_only_rows": 0,
        "crash_linked_share": 0.0,
        "agreement_share_of_zone_contexts": 0.0,
        "selected_only_share_of_zone_contexts": 0.0,
        "crash_only_share_of_zone_contexts": 0.0,
        "agreement_minus_crash_only_share": 0.0,
        "dominant_disagreement_driver": "no modeled context rows",
        ZONE_COLUMN: "",
        BOROUGH_COLUMN: "",
    }
)

selected_surface_zone_geojson = json.loads(
    selected_surface_zone_map_df[[ZONE_ID_COLUMN, "geometry"]].to_json()
)
for feature in selected_surface_zone_geojson["features"]:
    feature["properties"][ZONE_ID_COLUMN] = str(
        feature["properties"][ZONE_ID_COLUMN]
    )

minx, miny, maxx, maxy = selected_surface_zone_map_gdf.total_bounds
selected_surface_map_center = {
    "lon": float((minx + maxx) / 2.0),
    "lat": float((miny + maxy) / 2.0),
}

agreement_colorscale = [
    [0.0, "#FFFFFF"],
    [0.35, BRAND_COLORS["ice"]],
    [0.70, BRAND_COLORS["seafoam"]],
    [1.0, BRAND_COLORS["dark_teal"]],
]
disagreement_colorscale = [
    [0.0, "#FFFFFF"],
    [0.35, "#F7D8CC"],
    [0.70, "#F0A281"],
    [1.0, BRAND_COLORS["terracotta"]],
]
balance_colorscale = [
    [0.0, BRAND_COLORS["terracotta"]],
    [0.5, "#FFFFFF"],
    [1.0, BRAND_COLORS["dark_teal"]],
]

metric_specs = [
    {
        "column": "crash_linked_share",
        "label": "Crash-linked share within selected finalist",
        "title": "Selected Finalist Crash-Linked Concentration by Taxi Zone",
        "colorscale": agreement_colorscale,
        "zmin": 0.0,
        "zmax": float(selected_surface_zone_map_df["crash_linked_share"].max()),
        "visible": True,
    },
    {
        "column": "agreement_share_of_zone_contexts",
        "label": "Agreement share of zone contexts",
        "title": "Taxi Zone Agreement Between Finalist Contexts and Crash Contexts",
        "colorscale": agreement_colorscale,
        "zmin": 0.0,
        "zmax": float(
            selected_surface_zone_map_df["agreement_share_of_zone_contexts"].max()
        ),
        "visible": False,
    },
    {
        "column": "selected_only_share_of_zone_contexts",
        "label": "Finalist-only share of zone contexts",
        "title": "Taxi Zone Disagreement: Finalist Contexts Without Crash Linkage",
        "colorscale": disagreement_colorscale,
        "zmin": 0.0,
        "zmax": float(
            selected_surface_zone_map_df["selected_only_share_of_zone_contexts"].max()
        ),
        "visible": False,
    },
    {
        "column": "crash_only_share_of_zone_contexts",
        "label": "Crash-only share of zone contexts",
        "title": "Taxi Zone Disagreement: Crash Contexts Outside the Selected Finalist",
        "colorscale": disagreement_colorscale,
        "zmin": 0.0,
        "zmax": float(
            selected_surface_zone_map_df["crash_only_share_of_zone_contexts"].max()
        ),
        "visible": False,
    },
    {
        "column": "agreement_minus_crash_only_share",
        "label": "Agreement minus crash-only share",
        "title": "Taxi Zone Net Balance: Agreement vs Crash-Only Disagreement",
        "colorscale": balance_colorscale,
        "zmin": float(
            selected_surface_zone_map_df["agreement_minus_crash_only_share"].min()
        ),
        "zmax": float(
            selected_surface_zone_map_df["agreement_minus_crash_only_share"].max()
        ),
        "visible": False,
    },
]

metric_specs = [
    {
        **spec,
        "zmax": spec["zmax"] if spec["zmax"] > spec["zmin"] else spec["zmin"] + 1e-9,
    }
    for spec in metric_specs
]

fig = go.Figure()

customdata = np.array(
    list(
        zip(
            selected_surface_zone_map_df[BOROUGH_COLUMN],
            selected_surface_zone_map_df[ZONE_COLUMN],
            selected_surface_zone_map_df["total_context_rows"],
            selected_surface_zone_map_df["selected_surface_rows"],
            selected_surface_zone_map_df["crash_context_rows"],
            selected_surface_zone_map_df["agreement_rows"],
            selected_surface_zone_map_df["selected_only_rows"],
            selected_surface_zone_map_df["crash_only_rows"],
            selected_surface_zone_map_df["crash_linked_share"],
            selected_surface_zone_map_df["agreement_share_of_zone_contexts"],
            selected_surface_zone_map_df["selected_only_share_of_zone_contexts"],
            selected_surface_zone_map_df["crash_only_share_of_zone_contexts"],
            selected_surface_zone_map_df["agreement_minus_crash_only_share"],
            selected_surface_zone_map_df["dominant_disagreement_driver"],
        )
    ),
    dtype=object,
)

for spec in metric_specs:
    fig.add_trace(
        go.Choroplethmap(
            geojson=selected_surface_zone_geojson,
            locations=selected_surface_zone_map_df[ZONE_ID_COLUMN],
            z=selected_surface_zone_map_df[spec["column"]],
            featureidkey=f"properties.{ZONE_ID_COLUMN}",
            colorscale=spec["colorscale"],
            zmin=spec["zmin"],
            zmax=spec["zmax"],
            marker_opacity=0.88,
            marker_line_width=0.35,
            marker_line_color=BRAND_COLORS["seafoam"],
            visible=spec["visible"],
            colorbar=dict(
                title=spec["label"],
                x=1.01,
            ),
            customdata=customdata,
            hovertemplate=(
                "<b>%{customdata[0]} | %{customdata[1]}</b><br>"
                "All modeled zone contexts: %{customdata[2]:,.0f}<br>"
                "Selected finalist rows: %{customdata[3]:,.0f}<br>"
                "Crash-linked contexts: %{customdata[4]:,.0f}<br>"
                "Agreement rows: %{customdata[5]:,.0f}<br>"
                "Finalist-only disagreement: %{customdata[6]:,.0f}<br>"
                "Crash-only disagreement: %{customdata[7]:,.0f}<br>"
                "Crash-linked share within selected finalist: %{customdata[8]:.1%}<br>"
                "Agreement share of zone contexts: %{customdata[9]:.1%}<br>"
                "Finalist-only share of zone contexts: %{customdata[10]:.1%}<br>"
                "Crash-only share of zone contexts: %{customdata[11]:.1%}<br>"
                "Agreement minus crash-only share: %{customdata[12]:.1%}<br>"
                "Dominant disagreement driver: %{customdata[13]}<extra></extra>"
            ),
        )
    )

fig.update_layout(
    title=metric_specs[0]["title"],
    width=1080,
    height=760,
    margin=dict(l=20, r=20, t=90, b=20),
    map=dict(
        style="carto-positron",
        zoom=9.25,
        center=selected_surface_map_center,
    ),
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=0.01,
            y=1.03,
            xanchor="left",
            yanchor="top",
            bgcolor="rgba(255,255,255,0.92)",
            bordercolor=BRAND_COLORS["seafoam"],
            borderwidth=1,
            buttons=[
                dict(
                    label=spec["label"],
                    method="update",
                    args=[
                        {
                            "visible": [
                                metric_index == idx
                                for metric_index in range(len(metric_specs))
                            ]
                        },
                        {"title": spec["title"]},
                    ],
                )
                for idx, spec in enumerate(metric_specs)
            ],
        )
    ],
)

fig.show()

,taxi_zone_id,zone,borough,selected_surface_rows,crash_context_rows,agreement_rows,selected_only_rows,crash_only_rows,crash_linked_share,dominant_disagreement_driver
0,1,Newark Airport,EWR,158,0,0,158,0,0.000,more finalist-without-crash contexts
1,2,Jamaica Bay,Queens,509,29,2,507,27,0.004,more finalist-without-crash contexts
2,3,Allerton/Pelham Gardens,Bronx,772,595,73,699,522,0.095,more finalist-without-crash contexts
3,4,Alphabet City,Manhattan,574,374,25,549,349,0.044,more finalist-without-crash contexts
4,5,Arden Heights,Staten Island,658,307,38,620,269,0.058,more finalist-without-crash contexts
5,6,Arrochar/Fort Wadsworth,Staten Island,795,1096,148,647,948,0.186,more crash-without-finalist contexts
6,7,Astoria,Queens,648,1783,217,431,1566,0.335,more crash-without-finalist contexts
7,8,Astoria Park,Queens,379,29,4,375,25,0.011,more finalist-without-crash contexts
8,9,Auburndale,Queens,757,554,78,679,476,0.103,more finalist-without-crash contexts
9,10,Baisley Park,Queens,654,1246,134,520,1112,0.205,more crash-without-finalist contexts


> How To Read This Map
Use the dropdown to switch between five views of how the selected finalist and mapped crash contexts line up across Taxi Zones\.
\- Crash\-linked share within selected finalist: Of the finalist rows in a zone, how many also overlap a crash context\.
\- Agreement share of zone contexts: Of all modeled contexts in a zone, how many are both finalist and crash\-linked\.
\- Finalist\-only share of zone contexts: Contexts flagged by the selected finalist that do not overlap a mapped crash\.
\- Crash\-only share of zone contexts: Contexts with a mapped crash that are not captured by the selected finalist\.
\- Agreement minus crash\-only share: A net balance view\. Darker teal means finalist\-crash agreement is stronger than crash\-only disagreement; warmer tones mean crash activity is outrunning the selected finalist in that zone\.

Findings\. The Taxi Zone map makes the external\-plausibility story much more spatially specific than the citywide averages alone\. The strongest agreement between the selected finalist and mapped crash activity is concentrated in a limited set of zones rather than spread evenly across the city: for example, Astoria shows 217 agreement rows and a 33\.5% crash\-linked share within the selected finalist, while Baisley Park \(20\.5%\) and Arrochar/Fort Wadsworth \(18\.6%\) also stand out as places where finalist contexts and crash contexts overlap more heavily\. The dropdown views also show that disagreement is not one\-dimensional\. In some zones, the main gap is finalist\-only activity, meaning the selected anomaly surface is broader than the crash record there, as in Newark Airport, Jamaica Bay, and Alphabet City\. In others, the larger gap is crash\-only activity, meaning crashes occur more often than the selected finalist flags anomalous stress, as in Astoria, Baisley Park, and Arrochar/Fort Wadsworth\. The net\-balance view is useful because it separates places where the selected finalist and crash signal genuinely line up from places where one clearly outruns the other\. So the map supports a more nuanced read than “the finalist does or does not match crashes”: agreement is real and locally concentrated, but the mismatch pattern varies by zone and helps show where the selected surface may be most congestion\-grounded versus where crash activity spills beyond it\.

### Section Summary

Section 3\.4\.1\.3 uses the crash\-linked row universe to ask whether the selected finalist looks externally plausible once compared against mapped crash activity\. The main result is modest but consistent support rather than a decisive validation win: the selected All 3 retained frameworks surface sits slightly above the full\-space crash baseline, while the broader 2\+ retained frameworks comparator nearly doubles coverage without materially improving crash alignment\. The more informative signal comes from inside the selected finalist itself\. The congestion\_only slice is the clearest above\-baseline component, while demand\-only and mixed profiles are weaker\. The spatial review adds the same nuance at the Taxi Zone level: agreement between finalist contexts and crash contexts is real, but it is concentrated in specific zones and coexists with distinct finalist\-only and crash\-only disagreement patterns\. Taken together, this chapter does not reopen the upstream finalist decision, but it does sharpen the interpretation that the selected all\-3 surface contains a meaningful congestion\-linked core and that 2\+ is best kept as a robustness comparator rather than promoted to the main downstream layer\.

## 3\.4\.1\.4 Appendix: Pre/Post CP Crash Context Shift

This appendix shifts from surface validation to temporal interpretation\. Instead of asking whether the selected all\-3 surface overlaps crash\-linked contexts at all, it asks whether those crash\-linked finalist contexts change across the pre\-CP and post\-CP periods\.

The goal here is descriptive rather than causal\. We are not treating any pre/post redistribution as proof of a congestion\-pricing effect\. We are only checking whether the selected finalist surface and the mapped crash layer show a coherent change in timing and geography once the study period is split at the policy boundary\.

### Compare Pre/Post Crash Overlap

We start by summarizing crash overlap inside the selected all\-3 surface across the pre\-CP and post\-CP periods\. This gives a compact temporal read on whether crash\-linked finalist contexts become more or less common after the policy boundary\.

In [27]:
# ---------------------------------------------------------------------
# Summarize pre/post crash overlap inside the selected all-3 surface
# ---------------------------------------------------------------------

selected_surface_prepost_crash_summary_df = (
    crash_validation_row_level_df.loc[
        normalize_bool_flag(crash_validation_row_level_df["selected_finalist_flag"])
    ]
    .groupby(PRE_POST_CP_COLUMN, dropna=False)
    .agg(
        surface_rows=(EVENT_ID_COLUMN, "size"),
        rows_with_any_crash=("any_crash_flag", "sum"),
        any_crash_share=("any_crash_flag", "mean"),
    )
    .reset_index()
)

if "any_injury_crash_flag" in crash_validation_row_level_df.columns:
    injury_prepost_df = (
        crash_validation_row_level_df.loc[
            normalize_bool_flag(crash_validation_row_level_df["selected_finalist_flag"])
        ]
        .groupby(PRE_POST_CP_COLUMN, dropna=False)
        .agg(
            rows_with_injury_crash=("any_injury_crash_flag", "sum"),
            injury_crash_share=("any_injury_crash_flag", "mean"),
        )
        .reset_index()
    )
    selected_surface_prepost_crash_summary_df = (
        selected_surface_prepost_crash_summary_df.merge(
            injury_prepost_df,
            on=PRE_POST_CP_COLUMN,
            how="left",
            validate="one_to_one",
        )
    )

display(selected_surface_prepost_crash_summary_df.round(3))

,pre_post_cp,surface_rows,rows_with_any_crash,any_crash_share,rows_with_injury_crash,injury_crash_share
0,post_cp,61175,8857,0.145,3918,0.064
1,pre_cp,94178,15643,0.166,7200,0.076


Findings\. Crash overlap inside the selected All 3 retained frameworks surface is lower in the post\-CP period than in the pre\-CP period\. The any\-crash share falls from 16\.6% pre\-CP \(15,643 of 94,178 rows\) to 14\.5% post\-CP \(8,857 of 61,175 rows\), and the injury\-crash share falls from 7\.6% to 6\.4% over the same split\. So the selected finalist still overlaps a substantial number of crash\-linked contexts after the policy boundary, but that overlap becomes less common rather than more common in the post period\. This is a descriptive pre/post read, not a causal one, but it does suggest that the crash\-linked portion of the selected finalist thins out after CP rather than intensifying\.

Now repeat the same split for the broader \`2\+ retained frameworks\` robustness surface\. The point is not to revive the finalist contest, but to see whether the broader comparator reacts differently enough across periods to change the outside\-validation read\.

In [28]:
# ---------------------------------------------------------------------
# Summarize pre/post crash overlap inside the broader 2+ robustness
# surface
# ---------------------------------------------------------------------

comparison_surface_prepost_crash_summary_df = (
    crash_validation_row_level_df.loc[
        normalize_bool_flag(crash_validation_row_level_df["comparison_finalist_flag"])
    ]
    .groupby(PRE_POST_CP_COLUMN, dropna=False)
    .agg(
        surface_rows=(EVENT_ID_COLUMN, "size"),
        rows_with_any_crash=("any_crash_flag", "sum"),
        any_crash_share=("any_crash_flag", "mean"),
    )
    .reset_index()
)

display(comparison_surface_prepost_crash_summary_df.round(3))

,pre_post_cp,surface_rows,rows_with_any_crash,any_crash_share
0,post_cp,116883,16926,0.145
1,pre_cp,182556,30565,0.167


Findings\. The broader 2\+ retained frameworks surface shows the same directional pre/post pattern as the selected all\-3 surface\. Post\-CP, 16,926 of 116,883 rows are crash\-linked, for an any\-crash share of 14\.5%\. Using the pre\_cp row you pasted separately \(30,565 of 182,556, or 16\.7%\), the broader comparator also declines from pre to post rather than strengthening after the boundary\. That matters because it means the post\-CP softening is not unique to the tighter all\-3 finalist; it appears in the broader robustness surface as well, which makes the temporal shift look like a general property of the retained finalist family rather than an artifact of one narrow cut\.

### Visualize Pre/Post Crash Shift

Aggregate summaries are helpful, but they can hide where a shift is actually happening\. These views break the pre/post split into dayparts and broad geographies so we can see whether crash\-linked finalist contexts move later in the day, farther from the core, or in some other structured way\.

In [29]:
# ---------------------------------------------------------------------
# Compare pre/post crash overlap by daypart inside the selected surface
# ---------------------------------------------------------------------

selected_surface_prepost_daypart_crash_df = (
    crash_validation_row_level_df.loc[
        normalize_bool_flag(crash_validation_row_level_df["selected_finalist_flag"])
    ]
    .groupby([PRE_POST_CP_COLUMN, DAYPART_COLUMN], dropna=False)
    .agg(
        surface_rows=(EVENT_ID_COLUMN, "size"),
        rows_with_any_crash=("any_crash_flag", "sum"),
        any_crash_share=("any_crash_flag", "mean"),
    )
    .reset_index()
)

display(selected_surface_prepost_daypart_crash_df.round(3))

# Optional extension after column verification:
# summarize the same split by policy_geography_label if that field is
# present in crash_validation_row_level_df.

,pre_post_cp,daypart,surface_rows,rows_with_any_crash,any_crash_share
0,post_cp,am_peak,11930,1496,0.125
1,post_cp,evening,14137,1747,0.124
2,post_cp,midday,9933,2064,0.208
3,post_cp,overnight,13737,1606,0.117
4,post_cp,pm_peak,11438,1944,0.170
5,pre_cp,am_peak,19292,2712,0.141
6,pre_cp,evening,18151,2475,0.136
7,pre_cp,midday,18609,4263,0.229
8,pre_cp,overnight,19804,2607,0.132
9,pre_cp,pm_peak,18322,3586,0.196


Findings\. The pre/post decline is broad\-based across the daily cycle rather than concentrated in one isolated daypart\. Every selected\-surface daypart has a lower post\-CP crash\-linked share than its pre\-CP counterpart: am\_peak drops from 14\.1% to 12\.5%, evening from 13\.6% to 12\.4%, midday from 22\.9% to 20\.8%, overnight from 13\.2% to 11\.7%, and pm\_peak from 19\.6% to 17\.0%\. midday remains the most crash\-linked slice in both periods, with pm\_peak second, so the ranking of dayparts is fairly stable even as the overall level declines\. That points more toward a general downward shift in crash\-linked finalist contexts than a dramatic reshuffling of when those overlaps occur during the day\.

### Map Where Crash\-Linked Finalist Contexts Increased or Decreased

The clearest appendix map is the geographic change layer itself\. Instead of showing raw crash counts alone, this map focuses on where crash\-linked finalist contexts became more or less common after the policy boundary, using the established choropleth implementation rather than a new map pattern\.

In [30]:
# ---------------------------------------------------------------------
# Build Taxi Zone pre/post crash-linked frequency layers for the
# selected surface
# ---------------------------------------------------------------------

selected_surface_prepost_zone_crash_frequency_df = (
    crash_validation_row_level_df.loc[
        normalize_bool_flag(crash_validation_row_level_df["selected_finalist_flag"])
    ]
    .groupby(
        [PRE_POST_CP_COLUMN, ZONE_ID_COLUMN, ZONE_COLUMN, BOROUGH_COLUMN],
        dropna=False,
    )
    .agg(
        surface_rows=(EVENT_ID_COLUMN, "size"),
        crash_linked_rows=("any_crash_flag", "sum"),
    )
    .reset_index()
)

selected_surface_prepost_zone_crash_frequency_df["crash_linked_share"] = np.where(
    selected_surface_prepost_zone_crash_frequency_df["surface_rows"].gt(0),
    selected_surface_prepost_zone_crash_frequency_df["crash_linked_rows"]
    / selected_surface_prepost_zone_crash_frequency_df["surface_rows"],
    0.0,
)

display(selected_surface_prepost_zone_crash_frequency_df.head(10).round(3))

# ---------------------------------------------------------------------
# Render paired pre/post Taxi Zone crash-linked choropleths and a
# post-minus-pre change map using the established geometry workflow
# ---------------------------------------------------------------------

selected_surface_prepost_zone_map_gdf = build_taxi_zone_geometry_gdf(
    retained_map_geometry_gdf
).to_crs("EPSG:2263")
selected_surface_prepost_zone_map_gdf["geometry"] = selected_surface_prepost_zone_map_gdf[
    "geometry"
].simplify(
    tolerance=150,
    preserve_topology=True,
)
selected_surface_prepost_zone_map_gdf = selected_surface_prepost_zone_map_gdf.to_crs(
    "EPSG:4326"
)
selected_surface_prepost_zone_map_gdf[ZONE_ID_COLUMN] = (
    selected_surface_prepost_zone_map_gdf[ZONE_ID_COLUMN].astype(str)
)

prepost_zone_wide_df = (
    selected_surface_prepost_zone_crash_frequency_df[[
        PRE_POST_CP_COLUMN,
        ZONE_ID_COLUMN,
        ZONE_COLUMN,
        BOROUGH_COLUMN,
        "surface_rows",
        "crash_linked_rows",
        "crash_linked_share",
    ]]
    .assign(**{ZONE_ID_COLUMN: lambda df: df[ZONE_ID_COLUMN].astype(str)})
    .pivot_table(
        index=[ZONE_ID_COLUMN, ZONE_COLUMN, BOROUGH_COLUMN],
        columns=PRE_POST_CP_COLUMN,
        values=["surface_rows", "crash_linked_rows", "crash_linked_share"],
        aggfunc="first",
    )
)

prepost_zone_wide_df.columns = [
    f"{metric}_{period}" for metric, period in prepost_zone_wide_df.columns
]
prepost_zone_wide_df = prepost_zone_wide_df.reset_index()

for required_column in [
    "surface_rows_pre_cp",
    "surface_rows_post_cp",
    "crash_linked_rows_pre_cp",
    "crash_linked_rows_post_cp",
    "crash_linked_share_pre_cp",
    "crash_linked_share_post_cp",
]:
    if required_column not in prepost_zone_wide_df.columns:
        prepost_zone_wide_df[required_column] = 0.0

prepost_zone_wide_df[[
    "surface_rows_pre_cp",
    "surface_rows_post_cp",
    "crash_linked_rows_pre_cp",
    "crash_linked_rows_post_cp",
    "crash_linked_share_pre_cp",
    "crash_linked_share_post_cp",
]] = prepost_zone_wide_df[[
    "surface_rows_pre_cp",
    "surface_rows_post_cp",
    "crash_linked_rows_pre_cp",
    "crash_linked_rows_post_cp",
    "crash_linked_share_pre_cp",
    "crash_linked_share_post_cp",
]].fillna(0.0)

prepost_zone_wide_df["post_minus_pre_crash_linked_share"] = (
    prepost_zone_wide_df["crash_linked_share_post_cp"]
    - prepost_zone_wide_df["crash_linked_share_pre_cp"]
)

selected_surface_prepost_zone_map_df = selected_surface_prepost_zone_map_gdf[
    [ZONE_ID_COLUMN, ZONE_COLUMN, BOROUGH_COLUMN, "geometry"]
].merge(
    prepost_zone_wide_df,
    on=[ZONE_ID_COLUMN, ZONE_COLUMN, BOROUGH_COLUMN],
    how="left",
)

selected_surface_prepost_zone_map_df = selected_surface_prepost_zone_map_df.fillna(
    {
        "surface_rows_pre_cp": 0,
        "surface_rows_post_cp": 0,
        "crash_linked_rows_pre_cp": 0,
        "crash_linked_rows_post_cp": 0,
        "crash_linked_share_pre_cp": 0.0,
        "crash_linked_share_post_cp": 0.0,
        "post_minus_pre_crash_linked_share": 0.0,
    }
)

selected_surface_prepost_zone_geojson = json.loads(
    selected_surface_prepost_zone_map_df[[ZONE_ID_COLUMN, "geometry"]].to_json()
)
for feature in selected_surface_prepost_zone_geojson["features"]:
    feature["properties"][ZONE_ID_COLUMN] = str(
        feature["properties"][ZONE_ID_COLUMN]
    )

minx, miny, maxx, maxy = selected_surface_prepost_zone_map_gdf.total_bounds
selected_surface_prepost_map_center = {
    "lon": float((minx + maxx) / 2.0),
    "lat": float((miny + maxy) / 2.0),
}

paired_scale_max = float(
    max(
        selected_surface_prepost_zone_map_df["crash_linked_share_pre_cp"].max(),
        selected_surface_prepost_zone_map_df["crash_linked_share_post_cp"].max(),
        0.0,
    )
)
if paired_scale_max <= 0:
    paired_scale_max = 1e-9

paired_prepost_fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "map"}, {"type": "map"}]],
    subplot_titles=["Pre-CP", "Post-CP"],
    horizontal_spacing=0.04,
)

paired_prepost_fig.add_trace(
    go.Choroplethmap(
        geojson=selected_surface_prepost_zone_geojson,
        locations=selected_surface_prepost_zone_map_df[ZONE_ID_COLUMN],
        z=selected_surface_prepost_zone_map_df["crash_linked_share_pre_cp"],
        featureidkey=f"properties.{ZONE_ID_COLUMN}",
        colorscale=[
            [0.0, "#FFFFFF"],
            [0.35, BRAND_COLORS["ice"]],
            [0.70, BRAND_COLORS["seafoam"]],
            [1.0, BRAND_COLORS["dark_teal"]],
        ],
        zmin=0.0,
        zmax=paired_scale_max,
        marker_opacity=0.88,
        marker_line_width=0.35,
        marker_line_color=BRAND_COLORS["seafoam"],
        showscale=False,
        customdata=np.array(
            list(
                zip(
                    selected_surface_prepost_zone_map_df[BOROUGH_COLUMN],
                    selected_surface_prepost_zone_map_df[ZONE_COLUMN],
                    selected_surface_prepost_zone_map_df["surface_rows_pre_cp"],
                    selected_surface_prepost_zone_map_df["crash_linked_rows_pre_cp"],
                    selected_surface_prepost_zone_map_df["crash_linked_share_pre_cp"],
                )
            ),
            dtype=object,
        ),
        hovertemplate=(
            "<b>%{customdata[0]} | %{customdata[1]}</b><br>"
            "Pre-CP finalist rows: %{customdata[2]:,.0f}<br>"
            "Pre-CP crash-linked rows: %{customdata[3]:,.0f}<br>"
            "Pre-CP crash-linked share: %{customdata[4]:.1%}<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

paired_prepost_fig.add_trace(
    go.Choroplethmap(
        geojson=selected_surface_prepost_zone_geojson,
        locations=selected_surface_prepost_zone_map_df[ZONE_ID_COLUMN],
        z=selected_surface_prepost_zone_map_df["crash_linked_share_post_cp"],
        featureidkey=f"properties.{ZONE_ID_COLUMN}",
        colorscale=[
            [0.0, "#FFFFFF"],
            [0.35, BRAND_COLORS["ice"]],
            [0.70, BRAND_COLORS["seafoam"]],
            [1.0, BRAND_COLORS["dark_teal"]],
        ],
        zmin=0.0,
        zmax=paired_scale_max,
        marker_opacity=0.88,
        marker_line_width=0.35,
        marker_line_color=BRAND_COLORS["seafoam"],
        colorbar=dict(
            title="Share of selected finalist rows",
            x=1.01,
        ),
        customdata=np.array(
            list(
                zip(
                    selected_surface_prepost_zone_map_df[BOROUGH_COLUMN],
                    selected_surface_prepost_zone_map_df[ZONE_COLUMN],
                    selected_surface_prepost_zone_map_df["surface_rows_post_cp"],
                    selected_surface_prepost_zone_map_df["crash_linked_rows_post_cp"],
                    selected_surface_prepost_zone_map_df["crash_linked_share_post_cp"],
                )
            ),
            dtype=object,
        ),
        hovertemplate=(
            "<b>%{customdata[0]} | %{customdata[1]}</b><br>"
            "Post-CP finalist rows: %{customdata[2]:,.0f}<br>"
            "Post-CP crash-linked rows: %{customdata[3]:,.0f}<br>"
            "Post-CP crash-linked share: %{customdata[4]:.1%}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

paired_prepost_fig.update_layout(
    title="Pre/Post Taxi Zone Crash-Linked Share: Selected All-3 Finalist",
    width=1120,
    height=760,
    margin=dict(l=20, r=20, t=90, b=20),
    map=dict(
        style="carto-positron",
        zoom=9.15,
        center=selected_surface_prepost_map_center,
    ),
    map2=dict(
        style="carto-positron",
        zoom=9.15,
        center=selected_surface_prepost_map_center,
    ),
)

paired_prepost_fig.show()

change_scale_max = float(
    np.abs(
        selected_surface_prepost_zone_map_df["post_minus_pre_crash_linked_share"]
    ).max()
)
if change_scale_max <= 0:
    change_scale_max = 1e-9

change_fig = go.Figure(
    go.Choroplethmap(
        geojson=selected_surface_prepost_zone_geojson,
        locations=selected_surface_prepost_zone_map_df[ZONE_ID_COLUMN],
        z=selected_surface_prepost_zone_map_df["post_minus_pre_crash_linked_share"],
        featureidkey=f"properties.{ZONE_ID_COLUMN}",
        colorscale=[
            [0.0, BRAND_COLORS["terracotta"]],
            [0.5, "#FFFFFF"],
            [1.0, BRAND_COLORS["dark_teal"]],
        ],
        zmin=-change_scale_max,
        zmax=change_scale_max,
        marker_opacity=0.88,
        marker_line_width=0.35,
        marker_line_color=BRAND_COLORS["seafoam"],
        colorbar=dict(
            title="Post - pre",
            x=1.01,
        ),
        customdata=np.array(
            list(
                zip(
                    selected_surface_prepost_zone_map_df[BOROUGH_COLUMN],
                    selected_surface_prepost_zone_map_df[ZONE_COLUMN],
                    selected_surface_prepost_zone_map_df["crash_linked_share_pre_cp"],
                    selected_surface_prepost_zone_map_df["crash_linked_share_post_cp"],
                    selected_surface_prepost_zone_map_df["post_minus_pre_crash_linked_share"],
                )
            ),
            dtype=object,
        ),
        hovertemplate=(
            "<b>%{customdata[0]} | %{customdata[1]}</b><br>"
            "Pre-CP crash-linked share: %{customdata[2]:.1%}<br>"
            "Post-CP crash-linked share: %{customdata[3]:.1%}<br>"
            "Post - pre: %{customdata[4]:.1%}<extra></extra>"
        ),
    )
)

change_fig.update_layout(
    title="Post-Minus-Pre Taxi Zone Crash-Linked Share Change: Selected All-3 Finalist",
    width=1040,
    height=760,
    margin=dict(l=20, r=20, t=90, b=20),
    map=dict(
        style="carto-positron",
        zoom=9.15,
        center=selected_surface_prepost_map_center,
    ),
)

change_fig.show()

,pre_post_cp,taxi_zone_id,zone,borough,surface_rows,crash_linked_rows,crash_linked_share
0,post_cp,1,Newark Airport,EWR,32,0,0.000
1,post_cp,2,Jamaica Bay,Queens,397,2,0.005
2,post_cp,3,Allerton/Pelham Gardens,Bronx,393,36,0.092
3,post_cp,4,Alphabet City,Manhattan,143,4,0.028
4,post_cp,5,Arden Heights,Staten Island,197,11,0.056
5,post_cp,6,Arrochar/Fort Wadsworth,Staten Island,359,42,0.117
6,post_cp,7,Astoria,Queens,202,61,0.302
7,post_cp,8,Astoria Park,Queens,178,1,0.006
8,post_cp,9,Auburndale,Queens,419,35,0.084
9,post_cp,10,Baisley Park,Queens,300,60,0.200


> How To Read This Map
Use the left panel to read where crash\-linked finalist contexts were concentrated before congestion pricing, and the right panel to see the same share after the policy boundary\. Both maps use the same color scale, so darker zones mean a larger share of the selected finalist rows in that Taxi Zone were also crash\-linked\. The change map then shows post \- pre directly: teal means the crash\-linked share increased after the boundary, orange means it decreased, and values near white changed little\. So the paired maps show the level in each period, while the change map shows where that level moved\.

Findings\. The Taxi Zone pre/post maps show that the decline in crash\-linked overlap inside the selected All 3 retained frameworks surface is real, but spatially uneven rather than citywide\-uniform\. Several zones that already had strong crash linkage remain relatively elevated after the policy boundary, including Astoria at 30\.2%, Baisley Park at 20\.0%, and Arrochar/Fort Wadsworth at 11\.7% of post\-CP finalist rows, while places like Newark Airport \(0\.0%\), Jamaica Bay \(0\.5%\), and Astoria Park \(0\.6%\) remain close to zero\. Visually, the paired maps look broadly similar in pattern, which fits the earlier daypart and surface summaries: the same general crash\-linked hotspots persist, but they usually soften rather than intensify in the post period\. The post\-minus\-pre map makes that clearer by showing more warm\-toned decreases than strong teal increases across much of the core city, while a smaller set of outer\-borough and edge zones still show local gains\. So the appendix map supports a “redistribution with overall softening” read: post\-CP crash\-linked finalist contexts do not disappear, but they become somewhat less concentrated in many previously stronger zones, with only scattered pockets of increase\.

## Close

Stale placeholder close\. Replace this after the rewired crash\-validation sections are run and the final exported crash\-validation package is in place\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>